# Adaptive Hospital Room | Three-Model Master Reliability Screen
**Single Google Colab notebook · 9 October 2026 · development/model-selection only**

This notebook tests **GPT-6 Luna (OpenAI), Gemini 3.8 Flash (Google AI Studio), and Qwen3.8 27B (Cerebras)** through OpenRouter, using the **same five original WESAD-derived `PILOT_201–205` cases**. Each model attempts one non-agentic baseline and one bounded agentic workflow per case: **10 workflows/model, 30 total if all routes work**. Prompts, case adaptation, evidence, tool definitions, output schema, deterministic checks, and agent revision policy are shared; model-specific settings and exactly one provider are recorded separately.

**Purpose:** determine which model reliably completes the workflows without provider errors, malformed JSON, tool-contract violations, bad calculations, or constraint/evidence-reference validation errors. **No nurse assessment or recommendation appropriateness scoring in this notebook.** Existing historical Qwen results are not overwritten.

## Execution stages (run ALL cells for each stage)

1. `RUN_STAGE='offline'`, `APPROVE_PAID_CALLS=False`: validate 5 source-case hashes, shared implementation, 35+ regression checks, model-specific first-request cost reservations. **Never sends inference requests or requires an API key.**
2. `RUN_STAGE='smoke'`, `APPROVE_PAID_CALLS=True`: for each enabled model in sequence, check *live* OpenRouter endpoints and attempt **PILOT_201 baseline and agent**. No live generation if provider/tool/price checks fail. Inspect the status and costs before proceeding.
3. `RUN_STAGE='full'`, `APPROVE_PAID_CALLS=True`: **only models with saved successful smoke workflows** may proceed. The notebook verifies their original identities and reuses those two saved results, then attempts the remaining 4 cases × 2 workflows. Any failure is retained, never rescored, and provider failures stop that model, not the other models.

**Keep the same Colab runtime** for smoke → full, or enable Google Drive storage *before the first run*. To resume in a fresh runtime, restore the full saved results ZIP as documented in the notes. Never discard logs or rerun an unsuccessful paid workflow under the same identity just to improve the percentage.

### Fixed-provider routing and price safeguards

- GPT-6 Luna → OpenAI endpoint; reasoning `high`; no temperature override.
- Gemini 3.8 Flash → Google AI Studio endpoint; reasoning `high`; temperature 1.
- Qwen3.8 27B → Cerebras endpoint; provider-default reasoning; temperature 1.
- Every request uses a **single exact endpoint tag returned by live OpenRouter metadata**, `only=[tag]`, `allow_fallbacks=False`, and `require_parameters=True`. No DeepSeek models or providers.
- Provider headquarters are US-based; this does **not** independently verify where inference runs or prove US/EU data residency.
- The Cerebras provider page currently doesn't list Qwen despite Qwen's model listing referring to Cerebras. The notebook **must fail closed** if it cannot verify the Qwen+Cerebras endpoint, and will not silently substitute Darkbloom, DeepInfra or any other route.

**Local USD ceilings:** Luna $0.75, Gemini $3.50, Qwen/Cerebras $3.00, **global maximum $7.25 across all three**. These are conservative request reservations, not a guarantee of the provider's actual invoice or your OpenRouter remaining balance. 120 seconds is a **per-HTTP-request** timeout, not a full-workflow deadline. The model has up to six LLM calls, eight tool executions and one final-only revision. At most one transient retry per HTTP request.

## Provenance

The original input export (`AHR_PILOT_201_205_original.json`) and manifest are embedded byte-for-byte, and both their overall and five per-case SHA-256 values are verified before testing. The *recorded* HR/RMSSD/EDA measurements are WESAD-derived; room, time, preference and staff contexts are hypothetical. PILOT_203 intentionally contains unavailable measures; PILOT_205 has **locked lighting and no active procedure**. No WESAD participant labels or researcher-only provenance are supplied in model prompts.

**Model references:** https://openrouter.ai/openai/gpt-6-luna · https://openrouter.ai/google/gemini-3.8-flash · https://openrouter.ai/qwen/qwen3.8-27b · https://openrouter.ai/provider/cerebras


In [ ]:
# 1 · Install small dependencies (no GPU, local model or repository required)
%pip -q install "requests>=2.32,<3" "pandas>=2.2,<3" "jsonschema>=4.23,<5"

In [ ]:
# 2 · Experiment control panel — these are the only normal settings to change
RUN_STAGE = 'offline'                 # 'offline' -> 'smoke' -> 'full'
APPROVE_PAID_CALLS = False            # explicit authorization is required for BOTH live stages
ENABLED_MODELS = ['luna','gemini','qwen']  # sequential; remove a model to skip it
ALLOW_INVALID_SMOKE_FOR_FULL = False  # default: reject further paid runs if either smoke response was invalid

# Storage: in one Colab runtime, smoke results are reused automatically by the full stage.
USE_GOOGLE_DRIVE = False              # set True at the start if you want persistence between runtime resets
RESTORE_PREVIOUS_ZIP = None           # optional: full path of exported master-results ZIP uploaded to Colab
DOWNLOAD_ZIP = True

PROFILES = {
 'luna': {'model':'openai/gpt-6-luna','provider_name':'OpenAI','budget_usd':0.75,
          'per_request_cap':0.08,'max_input_price_per_m':0.15,'max_output_price_per_m':0.75,
          'reasoning':'high','send_temperature':False,'region_basis':'Provider headquarters: United States'},
 'gemini': {'model':'google/gemini-3.8-flash','provider_name':'Google AI Studio','budget_usd':3.50,
          'per_request_cap':0.35,'max_input_price_per_m':1.00,'max_output_price_per_m':4.50,
          'reasoning':'high','send_temperature':True,'region_basis':'Provider headquarters: United States'},
 'qwen': {'model':'qwen/qwen3.8-27b','provider_name':'Cerebras','budget_usd':3.00,
          'per_request_cap':0.25,'max_input_price_per_m':1.15,'max_output_price_per_m':1.90,
          'reasoning':None,'send_temperature':True,'region_basis':'Provider headquarters: United States'},
}
GLOBAL_BUDGET_USD = 7.25
MASTER_RUN_LABEL = 'AHR_WESAD5_Three_Model_Master_v1'

assert RUN_STAGE in ('offline','smoke','full')
assert type(APPROVE_PAID_CALLS) is bool and type(ALLOW_INVALID_SMOKE_FOR_FULL) is bool
assert type(USE_GOOGLE_DRIVE) is bool and type(DOWNLOAD_ZIP) is bool
assert isinstance(ENABLED_MODELS,list) and all(m in PROFILES for m in ENABLED_MODELS) and len(set(ENABLED_MODELS))==len(ENABLED_MODELS)
assert 'deepseek' not in str(PROFILES).lower()

# Common implementation initially binds to Luna for offline checks; the execution cell
# switches all model-specific globals atomically for EACH profile.
MODEL_CHOICE='luna'
P=PROFILES[MODEL_CHOICE]
MODEL=P['model']; PROVIDER=P['provider_name']; SEND_TEMPERATURE=P['send_temperature']
RUN_LIVE = RUN_STAGE != 'offline'
RUN_LABEL = MASTER_RUN_LABEL+'_shared'
CASE_IDS = ['PILOT_201','PILOT_202','PILOT_203','PILOT_204','PILOT_205']
REPEATS = 1
REASONING_EFFORT=P['reasoning']; TEMPERATURE=1.0; MAX_OUTPUT_TOKENS=16384
AGENT_MAX_CALLS=6; AGENT_MAX_TOOLS=8; TRANSIENT_RETRIES=1
REQUEST_TIMEOUT=120; PACE_SECONDS=5
RUN_BUDGET_USD=P['budget_usd']; MAX_REQUEST_RESERVATION_USD=P['per_request_cap']
MAX_HTTP_REQUESTS=70; MAX_ESTIMATED_INPUT_TOKENS=100000
MAX_INPUT_PRICE_PER_M=P['max_input_price_per_m']; MAX_OUTPUT_PRICE_PER_M=P['max_output_price_per_m']
CASES_JSON_PATH=None  # embedded original cases only
SOURCE_SHA256_EXPECTED = '215b8ee570c12219f031a840917032020ac1f67191388528c1144bf4f3d6d489'
SOURCE_CASE_SHA256_EXPECTED = {'PILOT_201': 'aaee37fa9f2ce2b9bd176887b7266d6d847ec684a1b6530d53486e8332bc8f58', 'PILOT_202': 'e7996adfdb2b91cc8c5e271740409c8533f3e47e01c3266f3793b11578e9d275', 'PILOT_203': '219b2beedd354dafbbd7a5c515e455a8464a88ef99391ba817ec3e1738229566', 'PILOT_204': '91cac71f86faa67dda8b81751c3146dc8aebd5f84c8b531ab3b5cc4dbe36cad1', 'PILOT_205': '86b850f3a0242b3b3586b26d25757ba59eaab263265a5d9f09e4cc4d0f296f8c'}
print('STAGE:',RUN_STAGE,'| models:',ENABLED_MODELS,'| total local cap: $',GLOBAL_BUDGET_USD)
print('Paid calls:', 'PERMITTED after checks' if RUN_LIVE and APPROVE_PAID_CALLS else 'OFF')


In [ ]:
# 3 · Imports and output location
import copy, csv, hashlib, json, math, os, re, time, uuid, zipfile
from datetime import datetime, timezone
from pathlib import Path
from contextlib import contextmanager
import types
import requests
import pandas as pd
from jsonschema import Draft202012Validator
from IPython.display import display, Markdown

def stamp(): return datetime.now(timezone.utc).isoformat()
def encode(x): return json.dumps(x, ensure_ascii=False, sort_keys=True, allow_nan=False)
def digest(x): return hashlib.sha256(encode(x).encode()).hexdigest()
def save_json(path, obj):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    tmp.replace(path)

if RUN_LIVE and not APPROVE_PAID_CALLS:
    raise ValueError('APPROVE_PAID_CALLS=True is required before a smoke/full live stage.')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/AHR_fresh')
else:
    BASE=Path('/content/AHR_fresh') if Path('/content').exists() else Path.cwd()/'AHR_fresh_results'
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_LABEL), 'Use letters, digits, _ and - in RUN_LABEL.'
MASTER_ROOT=BASE/MASTER_RUN_LABEL
MASTER_ROOT.mkdir(parents=True,exist_ok=True)
# Restore BEFORE generating even the shared offline files. Restore never overwrites files,
# and the subsequent write_frozen() checks detect incompatible case, tool or model versions.
def restore_master_archive(path):
    if not path: return
    source=Path(path)
    if not source.is_file(): raise ValueError('Prior master ZIP missing; upload it first: '+str(source))
    with zipfile.ZipFile(source) as z:
        names=z.namelist()
        if not names or len(names)!=len(set(names)) or sum(i.file_size for i in z.infolist())>50_000_000:
            raise ValueError('Unexpected archive contents/size; restore refused')
        for i in z.infolist():
            parts=__import__('pathlib').PurePosixPath(i.filename)
            if parts.is_absolute() or '..' in parts.parts or chr(92) in i.filename or i.filename.endswith('generation.lock'):
                raise ValueError('Unsafe archived path: '+i.filename)
            if i.is_dir():continue
            dest=MASTER_ROOT.joinpath(*parts.parts)
            if not dest.resolve().is_relative_to(MASTER_ROOT.resolve()):raise ValueError('Archived path escapes master root')
            if dest.exists() and dest.read_bytes()!=z.read(i):
                raise ValueError('Resume conflict; existing file differs: '+str(dest))
            if not dest.exists():
                dest.parent.mkdir(parents=True,exist_ok=True)
                dest.write_bytes(z.read(i))
    print('Restored previous master ZIP; saved results will be hash-verified:',source.name)
restore_master_archive(RESTORE_PREVIOUS_ZIP)
OUT=MASTER_ROOT/'shared'
OUT.mkdir(parents=True,exist_ok=True)

def require(condition,message):
    if not condition: raise ValueError(message)

def validate_settings():
    for key in ['MAX_OUTPUT_TOKENS','AGENT_MAX_CALLS','AGENT_MAX_TOOLS','MAX_HTTP_REQUESTS','REPEATS','MAX_ESTIMATED_INPUT_TOKENS']:
        value=globals()[key]; require(type(value) is int and value>0, key+' must be a positive integer.')
    require(type(TRANSIENT_RETRIES) is int and 0<=TRANSIENT_RETRIES<=1,'At most one transient retry is supported.')
    for key in ['RUN_BUDGET_USD','MAX_REQUEST_RESERVATION_USD','MAX_INPUT_PRICE_PER_M','MAX_OUTPUT_PRICE_PER_M','REQUEST_TIMEOUT']:
        value=globals()[key];require(type(value) in [int,float] and math.isfinite(value) and value>0,key+' must be finite and positive.')
    require(type(PACE_SECONDS) in [int,float] and math.isfinite(PACE_SECONDS) and PACE_SECONDS>=0,'Invalid pacing.')
    require(type(TEMPERATURE) in [int,float] and math.isfinite(TEMPERATURE) and 0<=TEMPERATURE<=2,'Invalid temperature.')
    require(isinstance(MODEL,str) and '/' in MODEL,'Use an explicit model ID.')
    require(isinstance(PROVIDER,str) and bool(PROVIDER),'A fixed provider name is required.')
    require(isinstance(SEND_TEMPERATURE,bool),'Temperature inclusion must be Boolean.')
    require(MODEL in [v['model'] for v in PROFILES.values()] and 'deepseek' not in MODEL.lower(),'Disallowed or unknown model.')
    require(REASONING_EFFORT in [None,'minimal','low','medium','high'],'Unsupported reasoning effort.')
    require(type(RUN_LIVE) is bool and type(USE_GOOGLE_DRIVE) is bool and type(APPROVE_PAID_CALLS) is bool,'Live/approval/storage switches must be Boolean.')
    require(not RUN_LIVE or APPROVE_PAID_CALLS,'Paid generation requires APPROVE_PAID_CALLS=True and RUN_LIVE=True.')
    require(isinstance(CASE_IDS,list) and CASE_IDS and all(isinstance(x,str) for x in CASE_IDS) and len(CASE_IDS)==len(set(CASE_IDS)),'CASE_IDS must be unique.')
    if MODEL.startswith('google/gemini-3'):
        require(TEMPERATURE==1.0,'Gemini 3: use temperature 1.0 to avoid the documented low-temperature looping risk.')
validate_settings()

def write_frozen(path,value):
    path=Path(path)
    if path.exists():
        require(encode(json.loads(path.read_text()))==encode(value),'Existing artifact differs: '+path.name+'. Use a new RUN_LABEL.')
    else: save_json(path,value)

# This guard runs before any case/evidence export, in offline and live modes.
RUN_IDENTITY={'version':'AHR-WESAD5-three-model-screen-1.0', 'settings':{k:globals()[k] for k in
 ['MODEL','PROVIDER','SEND_TEMPERATURE','CASE_IDS','REPEATS','REASONING_EFFORT','TEMPERATURE','MAX_OUTPUT_TOKENS','AGENT_MAX_CALLS',
  'AGENT_MAX_TOOLS','TRANSIENT_RETRIES','REQUEST_TIMEOUT','PACE_SECONDS','RUN_BUDGET_USD',
  'MAX_REQUEST_RESERVATION_USD','MAX_HTTP_REQUESTS','MAX_ESTIMATED_INPUT_TOKENS','MAX_INPUT_PRICE_PER_M','MAX_OUTPUT_PRICE_PER_M']},
 'external_cases_hash':SOURCE_SHA256_EXPECTED,'source_case_sha256s':SOURCE_CASE_SHA256_EXPECTED}
if not (OUT/'run_identity.json').exists():
    require(not (OUT/'manifest.json').exists() and not (OUT/'cost_ledger.json').exists(),
            'Legacy run folder found. Choose a new RUN_LABEL; do not overwrite it.')
write_frozen(OUT/'run_identity.json',RUN_IDENTITY)

@contextmanager
def generation_lock(directory):
    lock=Path(directory)/'generation.lock'
    try: fd=os.open(str(lock),os.O_CREAT|os.O_EXCL|os.O_WRONLY,0o600)
    except FileExistsError: raise RuntimeError('Another or interrupted run owns generation.lock; inspect saved results before recovery.')
    try:
        os.write(fd,stamp().encode());os.close(fd)
        yield
    finally: lock.unlink(missing_ok=True)

print('Master results:', MASTER_ROOT)
print(('LIVE '+RUN_STAGE.upper()) if RUN_LIVE else 'OFFLINE — no model calls')

## Evidence table: 12 proposal sources, plus optional action bases and calculation conventions
The table below records the population/exposure, supported findings, limitations, source-check depth and proposed AHR use. Most entries were checked against author abstracts; selected full-text sections are identified explicitly. This is a source-checked **development draft**, not an independent risk-of-bias review or a formally approved evidence base.

**Action support** means a researcher-authored applicability mapping, not a study prescription. **Context only** sources inform interpretation and caveats; they cannot alone justify an action. E04/E05 are optional action bases; E06 specifies calculation conventions. Staff and patient restrictions are mandatory. Optional bases do not impose extra restrictions or establish efficacy. The 10-point brightness step, 20% playback volume, 15-minute duration and night cutoff remain unvalidated simulation choices.

The baseline receives every summary. The agent receives a source index and can retrieve exactly those same summaries. Neither receives full papers through the tool. Both receive the same case, prototype rules and substantive instructions. This difference in evidence access is part of the workflow comparison and must be reported.

Review this table before another paid run. The notebook exports it as JSON, CSV and Markdown alongside a manual review rubric. Generation remains disabled.


**Version 1.4 policy correction:** Explicitly preferred nature sounds, music and white noise have equal preference-based support under E05. Research support remains a separate possible basis. Validation reports schema, calculations, action compliance and citation-contract validity separately for first and revised answers. Overall acceptance still requires all mechanical checks to pass; no historical output is rescored. Citation-contract validity checks source access and declared mappings, not factual accuracy of prose.


In [ ]:
# 4 · Versioned source summaries and separate prototype rules
EVIDENCE = [{'id': 'E01',
  'kind': 'research',
  'topic': 'lighting',
  'source': 'Giménez et al. (2017)',
  'proposal_key': 'gimenez2017lighting',
  'doi': '10.1111/jsr.12470',
  'url': 'https://doi.org/10.1111/jsr.12470',
  'verified_from': 'Publisher summary, methods and selected results sections',
  'checked_date': '2026-09-28',
  'study_design': 'Controlled clinical study; 196 cardiology patients',
  'exposure': 'Combined 24-hour lighting: brighter daytime exposure and lower night lighting over '
              'hospitalization.',
  'finding': 'Actigraphic sleep duration increased by about 5.9 minutes per hospitalization day; lighting '
             'appraisal improved. Subjective sleep, alertness, anxiety and depression did not differ between '
             'groups.',
  'limitations': 'Combined exposure over days does not isolate tone, establish acute calming, or validate '
                 'device percentages. One participant withdrew with intervention-induced migraine.',
  'ahr_application': 'Researcher mapping: day/night brightness direction may be considered; the prototype '
                     'clock cutoff and step size are not study prescriptions.',
  'role': 'action_support',
  'context_fields': ['lighting'],
  'lighting_actions': ['INCREASE', 'DECREASE', 'NO_CHANGE'],
  'sounds': [],
  'condition': 'time_of_day'},
 {'id': 'E02',
  'kind': 'research',
  'topic': 'sound',
  'source': 'Fan & Baharum (2024)',
  'proposal_key': 'fan2024natural',
  'doi': '10.1080/10253890.2024.2402519',
  'url': 'https://doi.org/10.1080/10253890.2024.2402519',
  'verified_from': 'Author abstract via PubMed/Europe PMC',
  'checked_date': '2026-09-28',
  'study_design': 'Systematic review and meta-analysis',
  'exposure': 'Natural sounds compared with quiet.',
  'finding': 'Differences favoured natural sounds for heart rate, blood pressure and respiratory rate; mean '
             'arterial pressure, perceived stress and oxygen saturation did not differ significantly.',
  'limitations': 'Mixed outcomes; abstract alone does not establish effects for this individual, a safe dose '
                 'or absence of adverse effects.',
  'ahr_application': 'Researcher mapping: nature sounds are an optional permitted sound, not a demonstrated '
                     'treatment for this case.',
  'role': 'action_support',
  'context_fields': ['sound'],
  'lighting_actions': [],
  'sounds': ['NATURE_SOUNDS'],
  'condition': 'sound_permission'},
 {'id': 'E03',
  'kind': 'research',
  'topic': 'physiology',
  'source': 'Shaffer & Ginsberg (2017)',
  'proposal_key': 'shaffer2017overview',
  'doi': '10.3389/fpubh.2017.00258',
  'url': 'https://doi.org/10.3389/fpubh.2017.00258',
  'verified_from': 'Open-access article; selected metric and interpretation sections',
  'checked_date': '2026-09-28',
  'study_design': 'Review of HRV metrics and norms',
  'exposure': 'Measurement duration, physiological influences and interpretation.',
  'finding': 'HRV interpretation depends on recording duration and context, including respiration.',
  'limitations': 'An RMSSD change alone does not establish psychological state or a patient-specific '
                 'uncertainty interval.',
  'ahr_application': 'Context for cautious descriptive interpretation; no lighting or sound efficacy claim.',
  'role': 'context_only',
  'context_fields': ['physiology'],
  'lighting_actions': [],
  'sounds': [],
  'condition': 'always'},
 {'id': 'E04',
  'kind': 'prototype_rule',
  'topic': 'lighting',
  'source': 'AHR prototype rule: retain brightness only',
  'url': None,
  'finding': 'E04 supports lighting.action=NO_CHANGE only: keep the current brightness. Within that choice, '
             'an allowed tone may be retained or selected using recorded context/preferences, unless '
             'lighting is locked. E04 does not support INCREASE or DECREASE.',
  'limitations': 'Permission to retain brightness does not show that existing lighting supports rest or '
                 'comfort. It is not evidence of a physiological benefit from any tone. For INCREASE or '
                 'DECREASE, omit E04 and use a supplied/retrieved applicable action-support source; '
                 'policy_ids may be empty.',
  'lighting_actions': ['NO_CHANGE'],
  'sounds': [],
  'condition': 'always',
  'role': 'prototype_rule',
  'context_fields': ['lighting']},
 {'id': 'E05',
  'kind': 'prototype_rule',
  'topic': 'sound',
  'source': 'AHR prototype design',
  'url': None,
  'finding': 'Silence, retaining permissible existing sound, or offering any explicitly preferred allowed '
             'sound may respect patient choice. All active sounds remain subject to staff permission and '
             'patient prohibitions.',
  'limitations': 'Preference compatibility is not efficacy evidence. A stated preference does not require '
                 'playback; explain a decision to retain silence when a permitted preference is supplied.',
  'lighting_actions': [],
  'sounds': ['NONE_SILENCE', 'NO_CHANGE', 'PREFERRED_CALMING_MUSIC', 'NATURE_SOUNDS', 'WHITE_NOISE'],
  'condition': 'preference_compatible',
  'role': 'prototype_rule',
  'context_fields': ['sound']},
 {'id': 'E06',
  'kind': 'prototype_rule',
  'topic': 'physiology',
  'source': 'AHR development convention',
  'url': None,
  'finding': 'Report signed current-minus-reference changes; HR/RMSSD percentages use the reference '
             'denominator. Report EDA absolute change only. Missing or poor-quality pairs are unavailable.',
  'limitations': 'These calculations are descriptive. EDA percentage reporting is disabled by design, not '
                 'because absolute comparisons are invalid.',
  'lighting_actions': [],
  'sounds': [],
  'condition': 'always',
  'role': 'prototype_rule',
  'context_fields': ['physiology']},
 {'id': 'E07',
  'kind': 'research',
  'topic': 'sound',
  'source': 'Papathanassoglou et al. (2025)',
  'proposal_key': 'papathanassoglou2025sound',
  'doi': '10.1016/j.aucc.2024.101148',
  'url': 'https://doi.org/10.1016/j.aucc.2024.101148',
  'verified_from': 'Author abstract via PubMed/Europe PMC',
  'checked_date': '2026-09-28',
  'study_design': 'Systematic review: 59 articles, 37 music and 22 nonmusic',
  'exposure': 'Randomized studies of music and nonmusic sound; only one direct comparison.',
  'finding': 'Reported benefits across pain, sleep and stress/anxiety outcomes; a larger music literature '
             'does not establish superiority over nonmusic sound.',
  'limitations': 'Heterogeneous interventions and outcomes; no fixed AHR playback dose or individual benefit '
                 'follows from the abstract.',
  'ahr_application': 'Researcher mapping: offer explicitly preferred calming music when permitted, without '
                     'promising benefit.',
  'role': 'action_support',
  'context_fields': ['sound'],
  'lighting_actions': [],
  'sounds': ['PREFERRED_CALMING_MUSIC'],
  'condition': 'sound_permission'},
 {'id': 'E08',
  'kind': 'research',
  'topic': 'sound',
  'source': 'Ding et al. (2025)',
  'proposal_key': 'ding2025whitenoise',
  'doi': '10.1016/j.sleep.2025.106869',
  'url': 'https://doi.org/10.1016/j.sleep.2025.106869',
  'verified_from': 'Author abstract via PubMed/Europe PMC',
  'checked_date': '2026-09-28',
  'study_design': 'Meta-analysis: 12 randomized trials, 1,301 participants across age groups',
  'exposure': 'White noise for sleep; adult, older-adult and infant populations.',
  'finding': 'Adult and older-adult pooled sleep-quality scores improved; adult results had substantial '
             'heterogeneity.',
  'limitations': 'Infant findings cannot simply transfer to adult hospital cases. Sleep outcomes do not '
                 'establish acute stress relief or an exact device volume.',
  'ahr_application': 'Researcher mapping: white noise may support an explicitly stated sleep/rest goal with '
                     'preference and permission.',
  'role': 'action_support',
  'context_fields': ['sound'],
  'lighting_actions': [],
  'sounds': ['WHITE_NOISE'],
  'condition': 'sleep_goal'},
 {'id': 'E09',
  'kind': 'research',
  'topic': 'sound',
  'source': 'Nien et al. (2026)',
  'proposal_key': 'nien2026whitenoise',
  'doi': '10.1016/j.iccn.2025.104239',
  'url': 'https://doi.org/10.1016/j.iccn.2025.104239',
  'verified_from': 'Author abstract via PubMed/Europe PMC',
  'checked_date': '2026-09-28',
  'study_design': 'Randomized trial: 58 ICU patients in Taiwan',
  'exposure': '40–50 dB white noise from 23:00 to 07:00 for three consecutive nights versus routine care.',
  'finding': 'Some self-reported sleep measures improved during follow-up.',
  'limitations': 'Small single-setting trial; overnight sound is not equivalent to a 15-minute simulation. '
                 'Volume percentage has no established conversion to decibels.',
  'ahr_application': 'Researcher mapping: optional preferred white noise for sleep/rest; do not copy '
                     'exposure as a validated AHR dose.',
  'role': 'action_support',
  'context_fields': ['sound'],
  'lighting_actions': [],
  'sounds': ['WHITE_NOISE'],
  'condition': 'sleep_goal'},
 {'id': 'E10',
  'kind': 'research',
  'topic': 'sound',
  'source': 'Saskovets et al. (2025)',
  'proposal_key': 'saskovets2025sound',
  'doi': '10.2196/69120',
  'url': 'https://doi.org/10.2196/69120',
  'verified_from': 'Author abstract via PubMed/Europe PMC',
  'checked_date': '2026-09-28',
  'study_design': 'Review: 34 clinical and laboratory studies',
  'exposure': 'Music and sound interventions across settings.',
  'finding': 'Physiological and self-reported responses vary with preferences, delivery and cultural '
             'context; some adverse responses were reported.',
  'limitations': 'Heterogeneous findings do not establish a universally beneficial or harmless sound.',
  'ahr_application': 'Context for preference-sensitive choice and cautious claims; not independent support '
                     'for a specific sound option.',
  'role': 'context_only',
  'context_fields': ['sound'],
  'lighting_actions': [],
  'sounds': [],
  'condition': 'always'},
 {'id': 'E11',
  'kind': 'research',
  'topic': 'lighting',
  'source': 'Martins et al. (2025)',
  'proposal_key': 'martins2025light',
  'doi': '10.1016/j.neubiorev.2025.106241',
  'url': 'https://doi.org/10.1016/j.neubiorev.2025.106241',
  'verified_from': 'Author abstract via PubMed/Europe PMC',
  'checked_date': '2026-09-28',
  'study_design': 'Systematic review: 25 studies, all assessed at high risk of bias',
  'exposure': 'Light exposure and vagally mediated HRV in healthy and clinical populations.',
  'finding': 'Higher-illuminance blue light generally corresponded to lower vmHRV; lower-illuminance warm '
             'light showed potential increases, with inconsistent findings.',
  'limitations': 'High risk of bias and heterogeneity prevent treating warm tone as proven calming or '
                 'diagnosing state from HRV.',
  'ahr_application': 'Context for uncertainty about tone; does not independently justify a brightness '
                     'direction or a numerical setting.',
  'role': 'context_only',
  'context_fields': ['lighting'],
  'lighting_actions': [],
  'sounds': [],
  'condition': 'always'},
 {'id': 'E12',
  'kind': 'research',
  'topic': 'lighting',
  'source': 'Linders et al. (2026)',
  'proposal_key': 'linders2026lighting',
  'doi': '10.1016/j.iccn.2026.104421',
  'url': 'https://doi.org/10.1016/j.iccn.2026.104421',
  'verified_from': 'Author abstract via PubMed/Europe PMC',
  'checked_date': '2026-09-28',
  'study_design': 'Systematic review: 28 ICU studies, 14 intervention studies',
  'exposure': 'Naturalistic and intervention lighting research in intensive care.',
  'finding': 'Marked heterogeneity prevented definitive outcome conclusions and clear lighting-design '
             'guidance.',
  'limitations': 'ICU evidence cannot establish universal parameters or a guaranteed sleep benefit.',
  'ahr_application': 'Context for cautious transfer and the absence of a validated AHR lighting dose.',
  'role': 'context_only',
  'context_fields': ['lighting'],
  'lighting_actions': [],
  'sounds': [],
  'condition': 'always'},
 {'id': 'E13',
  'kind': 'research',
  'topic': 'physiology',
  'source': 'Posada-Quintero & Chon (2020)',
  'proposal_key': 'posada2020innovations',
  'doi': '10.3390/s20020479',
  'url': 'https://doi.org/10.3390/s20020479',
  'verified_from': 'Author abstract via PubMed/Europe PMC',
  'checked_date': '2026-09-28',
  'study_design': 'Systematic review of electrodermal measurement and processing',
  'exposure': 'Tonic and phasic electrodermal activity and applications.',
  'finding': 'EDA reflects sympathetic sweat-gland activity and is used across multiple arousal-related '
             'applications.',
  'limitations': 'EDA alone is not a specific psychological stress diagnosis. The AHR absolute-only '
                 'reporting rule is a prototype convention.',
  'ahr_application': 'Context for descriptive EDA interpretation, not intervention efficacy.',
  'role': 'context_only',
  'context_fields': ['physiology'],
  'lighting_actions': [],
  'sounds': [],
  'condition': 'always'},
 {'id': 'E14',
  'kind': 'research',
  'topic': 'lighting/sound',
  'source': 'Mendonça et al. (2025)',
  'proposal_key': 'mendonca2025nursing',
  'doi': '10.1111/inr.13062',
  'url': 'https://doi.org/10.1111/inr.13062',
  'verified_from': 'Publisher abstract, eligibility and selected intervention-table sections',
  'checked_date': '2026-09-28',
  'study_design': 'Review: 13 studies, 1,975 hospitalized adults and older adults; noncritical wards',
  'exposure': 'Nursing interventions including environmental, sensory, educational and individualized care.',
  'finding': 'Reported sleep-focused approaches span multiple components, including light and noise '
             'management.',
  'limitations': 'Critical-care patients were excluded; multicomponent evidence cannot establish an isolated '
                 'light/sound effect or AHR dose.',
  'ahr_application': 'Context for individualized care and staff review; not independent support for a '
                     'particular model action.',
  'role': 'context_only',
  'context_fields': ['lighting', 'sound'],
  'lighting_actions': [],
  'sounds': [],
  'condition': 'always'},
 {'id': 'E15',
  'kind': 'research',
  'topic': 'lighting/sound',
  'source': 'Wang et al. (2025)',
  'proposal_key': 'wang2025nurse',
  'doi': '10.1111/nicc.70028',
  'url': 'https://doi.org/10.1111/nicc.70028',
  'verified_from': 'Publisher abstract',
  'checked_date': '2026-09-28',
  'study_design': 'Before/after ICU quality-improvement project',
  'exposure': 'Nurse-led SLEEP bundle including noise/light reduction, sensory aids and '
              'pharmacological/nonpharmacological measures.',
  'finding': 'Reported sleep questionnaire scores rose from 62 to 70 after implementation.',
  'limitations': 'Bundle and before/after design do not isolate lighting or sound causality. Medication '
                 'components are outside AHR.',
  'ahr_application': 'Context for staff-led multicomponent care, not evidence that this prototype reproduces '
                     'the bundle.',
  'role': 'context_only',
  'context_fields': ['lighting', 'sound'],
  'lighting_actions': [],
  'sounds': [],
  'condition': 'always'}]
# The legacy kind/ID fields remain for schema compatibility. Decision force is explicit.
for entry in EVIDENCE:
    if entry['id'] in ['E04','E05']:
        entry['decision_force']='OPTIONAL_ACTION_BASIS'
        entry['mandatory']=False
        entry['restricts_other_actions']=False
        entry['role']='optional_action_basis'
    elif entry['id']=='E06':
        entry['decision_force']='CALCULATION_CONVENTION'
        entry['role']='calculation_convention'
e04=next(e for e in EVIDENCE if e['id']=='E04')
e04['source']='AHR optional rationale: retain brightness'
e04['finding']='Retaining the current brightness is an available choice when compatible with the case. This entry supplies a non-research rationale for that choice. Tone is selected separately within the case restrictions.'
e04['limitations']='This entry imposes no lighting lock, default preference, veto, or requirement to retain brightness. Other choices may use another applicable basis. A chosen increase/decrease needs its own applicable support; do not cite this retention rationale for those actions. No physiological or sleep benefit is established.'
e05=next(e for e in EVIDENCE if e['id']=='E05')
e05['source']='AHR optional rationale: preference-compatible sound'

POLICY={"brightness_step":10, "sound_volume":20, "sound_duration_minutes":15,
        "night_start":19, "night_end":7, "comparison_tolerance":0.01,
        "meaning":"simulation settings, not clinically validated doses"}
METRICS={"hr_bpm":"bpm", "rmssd_ms":"ms", "eda_us":"microsiemens"}
TONES=["WARM","NEUTRAL","COOL"]
LIGHT_ACTIONS=["INCREASE","DECREASE","NO_CHANGE"]
SOUNDS=["PREFERRED_CALMING_MUSIC","NATURE_SOUNDS","WHITE_NOISE","NONE_SILENCE","NO_CHANGE"]

EVIDENCE_REVIEW_STATUS='SOURCE_CHECKED_DEVELOPMENT — researcher approval pending'
EVIDENCE_VERSION='proposal_sources_1_support_representation_4'
review_fields=['id','source','study_design','exposure','finding','limitations','ahr_application','role','verified_from','url']
evidence_review=pd.DataFrame([e for e in EVIDENCE if e['kind']=='research'])[review_fields]
display(evidence_review)
write_frozen(OUT/'evidence_table.json',{'version':EVIDENCE_VERSION,'status':EVIDENCE_REVIEW_STATUS,'entries':EVIDENCE,'simulation_policy':POLICY})
evidence_review.to_csv(OUT/'evidence_review.csv',index=False)
source_text=['# AHR development evidence review',EVIDENCE_REVIEW_STATUS,
 'Twelve decision-relevant sources cited in the proposal; not a new systematic review. Abstract-only entries are labelled. Action mappings are researcher-authored, not prescriptions from the papers.']
for entry in EVIDENCE:
    source_text.append('\n## '+entry['id']+' — '+entry['source'])
    for key in review_fields[2:]+['proposal_key','doi']:
        if key in entry: source_text.append('**'+key.replace('_',' ')+':** '+str(entry[key]))
_ = (OUT/'evidence_review.md').write_text('\n\n'.join(source_text))


In [ ]:
# 5 · Import five ORIGINAL source records with SHA-256 verification (no API calls)
# The two compressed literals below decompress to the exact bytes originally uploaded.
import base64, zlib
_ORIGINAL_CASES_ZLIB_B64 = (
    'eNrtnd1u28YSgO/zFAtdxzRJifqxr1xbSYw6ciDZzSmKgFiRK2kbimS5pBShKNB36LOcFziP0ic5M/wTRUl2EseNXE+BIrG4Ws7O'
    'zM7M7n7e/PKCsd/hf8YaypmJObcXIlIy8BsnrOGKhfCCcC782FYikkLZC7PxMmvucCVs6WK7d5dX1ze2qRvFM+mHSazgUdY1fBLO'
    'VtCrF0xXlU/h85ngUWxHPBb2OJxvPMN3JFEEL4eP202t1em9rD4cgwCe9EX2tGeaG08TX+L3GmPBY3U8l37j5Y6e7aX03WBpw/Mk'
    'FiixtfMdn91uysNKI32jkUrmcx6t7LmIZ0GquIFIouBHGZvMx799lDFLfCcA0ZxYuGzIQsE/qlPW1uE/dszmgvtsOGRz6XlSCSfw'
    'XbV7XL8l3JPxqqZQFCLmcYKyNV5fX19sfBmeRoKrzPavEs87ykbN+uevmYojGSrGfZe5Is7kc4J56IlPQjGVhGEQxSyeiY0RoPah'
    'WeLHp8wPGE/iYM5j6WQPhCtj6DIMPSlcrS5LMAafWwjXdgLwST5FQxuaXmuVKD72RLWND5LXGuUCgVvbuRFS7QdsyRdiEkRzJv1Y'
    'RGHgcWzEgggGGUl4+dGCe4lgE1S4P9XYWTkCVMq6X+aAZqQLfqzYUkQCRhuvR1aR5o/dfkP2OlR7vdhhuUY0V8q152p/wGoammnt'
    'DVhmR9NNa2fAmqunHKmGb0fgXI4jlAJzZGYCkzBXTiagZh8evETPgtDlgPL90n5rV1sb6TuGtkLw9VRxhXKgXYzDqg49lqDF6enO'
    'QamVDzOsMqh8iimNDdDfmAOmkA7oJ5qt4tmccffXxIUPsAMKiGTlf3kYFS6HGOREgZICqsy7wqnWMax90RQf9nYG0//9d3SI0fQt'
    '1nHgEHKMlS8k5plQMetfnDG0Z5qVJ2BO0P9CHGXdMuVJR6QOuGG0R4qR52vhJpWJFEfcESwSCymW8Ggp41kqMv6cTq2I+1MB0Z5P'
    '/UCBv6hUYjcKwiCJ0cEmIDZqjEkXBJUTdBQ2hFI2cqHDBY8kH0uUPe1OzTmoAd7qg4f4UHxEIubwbZy6qZqq8xWrYwqaZPZnEUVf'
    '1BwBxAQpPsWby2svgChjQ/LCsTVM/aSpN6rx11/IKPDTBf404B42GvZHN/boqt9/V20JkmIjDvIutj0MHmP2FG7RweXg9WZgymOj'
    'rSSUgNVu6gpvwNhTYd+swiBLq5A4Z4EKZQx/yUd5wkLQI0jEsjejh4Df+VPmJhH+gU6SB0OWOfEpEzwCjUKpmb8dEvxHP1j6FRVX'
    'k5PDI3SNba3idkYUOAJeJWzp2/DDFF6P02sCtYT4muHAbCn7ZFKBaOBDS77aI5sKkmhTjzVz5MU2vlnX9L///Kupw58s3zM4YVDP'
    'w/SCikRwF/UFbgk+i5VQ6oAgy0LizKyoMypmaza3dlQvlXz9T741hJADakvjF+gTPC1IFMs/hXgCJshmZqpXvuDSw2mvsddJ+rYx'
    'iBPP8IXQQIk0+o1XKE0cOIF3ytwgm5EK5r8oOxZg6uUM5K6JU5aDNoYVj6/swgfKeLfeDTsunREW5xAqogDKQwbyBRCFg2gFZWvi'
    'rjT2pvAYnMsvWRRAOyVidHlYUOHchT/C0u5Zncv9FStmZjlh5iAVTngoUT0I7eDkbFbxzVPU+gpCuZuO2eGJEviRqpgCIq4CDWAv'
    'quqgWzEJxawFJDmdodB1jx1H+MCHWWSHInIyJ7I266g4SKu9xqB/ezM8u9o7M3y33nu8CrNvXg/69ujyqj84728abRF4YNvKuzdf'
    'DYOF/ANy2+CaqTNVq707NJBr3a6YZlMh5brXLlRjOzPMpsqGHBgs05gaR0klqjSyziLIbul3bFQMdvtLVeT3Z8O3m2Pc0tuHnX2m'
    'GrRRZVt9Ds5uboegwevbwcVodz/Zt4MwtiHzb0dEjJwzOZbxtvAfdreqifNhS8VQ20wmNsb/SKbpVO12ORuS4cdUnXWZdjrf77X0'
    'heuMTaeY80/wYUff6Ye58b6hhTJF7HWK4oV3mW+v+3+GYWG09tYsqe5hpS12TI9moaBsduRKgikFoTZyZrgs4lD78OpxgYLaJ64f'
    'QYTSC+JS7gZ+x84SIbZ83x+dXdjvL2/e2KOfBzdv+jeX5/b59eCm/5+bypeKwjGrdy/W3bML4wRqN8hLUOFhNIbSLj5l2QswIkO8'
    'T6MqZilY/ftpRZyNQURHge+t1omgsW5kr2tvUO/wvG9fXJ/fvu2DYOtCPE/omxbb8MA842cHLu+NWt6VWYYr9VCInYVsKH0+QZ7J'
    'NzGyCjt7PuMKQjtW1cVQjpdCcdfOmuKsApOnBl0Yx3PuywnMM+1XVV94JlFaPM7iOFQnx8duILUgmh4bumYYLeu4abbMXrunpX92'
    're2c8WGH5vYrA5YPnmtDbJ3hS7PTJm2dVLXNUyWt3Cqor5qkWnvORX94+VP/opbLC63X5xJLjVD54MPOqu/CYGXgL30JK1WsfFJD'
    '5Es2WDI4Es/dNpY6hVHgL1OpoFbaXQ4+RDdFyfY9VJO/+0AUU+zpk7vs1Ao5SqGS+q4lOcyd2nnmjpOugXaP/c3P766zQgVqvjuG'
    'vnt45ZIwX2auzyGK4Zb7A7BUDDmGfA9GGzjJPN9Uc9J6ZwoFsQM/p4vPMsd/rSfsWPA8n9HvWIs8n8Hnm1laZdPxMQY/Ko/bZjU1'
    'PFDurd3QJyV9fYf2SQm/sdf6pCSv78T+41luvamY71eu97fTc5oEQnCEeS1e3bnsKjO4HURyKv39YhdHDZWG5Yp73QgaRDw7NilS'
    '6Oa+TKHQcuU5D1zh2bg3vMLVZp6ej4v0rOrLzoaacdNqYx+i1+NtMZm0jU7L1Lu61dE7HccQE6fTnuiW1XXafOJYHdPSO92J6LW6'
    'QvAx1x3L6pncbTa2dpVcSF9OXBUdTypeFMpLG34zcNJ8PHCyq2u9trnv5NzQO1q7ZxE5SeQkgUJkr4MgJw2jCyFrX8Rq9bRul8hJ'
    'IieJqSNyksjJ+8lJo61Bjbu3AGxq3V6H2EliJ4mdJLMTO7mXnTSME/1edvJ2MHrXP798dbmxyfMl8OTt4MfB9fvBQ+FJ0HAEbo0W'
    'Kje8jgoCrNwfyl6M1FsFihuJlPcq943X2Fd+hlJ8/Tlik2eF6uBdqaNlcCFO+QBzD3wMWoNZO6nAi+i/X8ZJnrEwGUPOOlK4OCiY'
    'DXQ1MCPMdyiF0qqp3PJbcijEIBO6iSfWuAdXH9kYcSCOW1Bp6EljgFQ5E5vSfzgO+EmEwsf44q3WoyLQkkBLAi23QcvNuLuXnaw1'
    'q6ORW70QGXkvGbnx4fn1NbGS34eVNE+K356oBkPCJQmXfBCxYxIuuV81hEsevrscAC5pEi75hBzmgHBJk3BJwiUJlyRcknBJwiUJ'
    'lyRc8lBxyeYj3jPZ1tpGa99p+dZ5C7GSxN79u864bwfD/tXl2Q9X/f1WQ6FcmOdgCxQqPQGZj+U0PQ7K7IYBiPEI91eZZf39519t'
    'PW1nNPEHo1MeduH24SK7yCU/r6wenqdGBZUcVyiiNTSUmjeCgk1GjLtBGBNbeQ9buRXAPie4EVf5cK5y17n+xpz6AXz5qBSu4hJK'
    '+ErmFQrOoAWoAVcieVhk8yAKZ2nGY9nhGagh3Yde5KfSoIDRxSl7M8xOXcvlTR5NKSZSTCRQ8p7YqGudTpMoyS+LeW8vR6P6RXCb'
    'xFx+QRuKVUEd6vgIhSfCEglL3IMldglLfIZYYsvaxAVzmG6acXYoRxR4HrjrGrmDFXTkwHI+m+piBY7peIHCuV+FBzWGfX838vFV'
    '4vsrhrdABvjLH2FGLTI1Q4/w8+sjlfgtyeC7dGrBwyXdCUmoIqGKdCckkY9EPj4u+dg8qeRXNpdKYZaAJQzBjwQ/Poi/aRL8uF81'
    'BD8evrscAPzYJPjxCTnMF8KPF/2ryx/6w7MbyIv9V/3zm2+roH6GYKSbV7jJE+KQcccJh1/uO8y5Sn8jj0BHAh0JdCTQkUBHAh0J'
    'dHzGoONdouPxqp0SjNUtioV5nFcRZT93jKE91q2J4ThG09StpmVZbseFtXG3p7dMPhk3m441bjp6W3S55eqi5bacXtsa9xwYrmO2'
    'eWNzY+Wb05itx6MxLUOz9l9eaXU0UAHxmMRj0lVtZK/DuLuyDRGr19x725qpa6ZBt1fS7ZV0ryHdXklQ5v1QJpR4vb2/jtPRDKIy'
    '6e5KuruSzE6Q6N13V97/736/P/ux/+r2atAfjb4SEn17/VMfWYXPpETNryErS1Jt77/zzfAaQ0jcy4Dlr4akDOYCFwSHgocau0Fa'
    'seQYWeEzVcQu/f0LuuHyEW+4/IbIKqGfhH4S+nk3WPiFGCGhn4R+EvpZQz9blfojT9AEfRL0+SAoq0XQ537VEPR5+O5yANBni6DP'
    'J+QwB3TjZYtuvCQQlEBQAkEJBCUQlEBQAkEP9cZL6xFvvDS1lrGXWGq3taZuEGNJjCUBRWSvg2AscZew093LBHW0rmkQYUmEJbF3'
    'RFgSYXkvYWlq1t7qD57RPw5OgCUBlmR2AizvAix7dAsn/ePghE4SOknoJN2a+YzQyZo6iZwkcnInOWmdsMxnWOFDBE4SOPkgdsci'
    'cHK/agicPHx3OQBw0iJw8gk5zAGBkxaBkwROEjhJ4CSBkwROEjhJ4OQ3BCdffHjxf3VysLk='
)
_ORIGINAL_MANIFEST_ZLIB_B64 = (
    'eNplU9tqGzEQfc9XCL/WNrqNNMpboIUGShPaQB9KMbqMmoXNyuyuTUvIv3c2SUmw3zSHOWfOXPR4IcSK/uzbOFPZHea8uhQrLbXb'
    'KLmReKfNpZaXUm2tNtrqD5IDuVovrL7FQuOSH+/HbaEj9W3/QMO8m2jsaNod9fbltV1SdzlONL1Ql+cut8MwMx3eoK5MDPzkmJHb'
    '6y83dzst1TPnHaBPAXMK2FMAVhz/eis03UcNji0Nbehy7Lnq42nVpbFIZHyNoepMOoVUlHeIPnntXHEFrafs0EaVHBhZwFh0hMbo'
    'lLECnjtnUfIhuFhqSayocsYMpL3yVloZMoIx1RALS5UNl6nGB5OUAo8UivZw3v2yMhVYjqgUA7bEmlLxETIoIAsQ0TobEamGYIJK'
    'ERUbN6S8Qa0DOHc+QRZldzF7VdHVGJ0vJWJiJrAxZV3JGCkVqGgzJjAqJpMgZ1sSGZdjUedbYFF0CUFWEyXfEzMMMKJd0eDBpwiB'
    'YuSYW4cIJVQZyC6isurgKuZlk0/rt7P9v8v0d6bpZRSQkAi8zErzXKpka8izVV4a3oGMWVXnVVAGETRmpaxN1VbDG7UYXk50aocx'
    '06J3M3a/uyH2om98KaJ2R9osRyR+fPp+9XHDf4CRIq4+fxPvPoHohv1hfr33ob16+9rEM3Ue4zDVNj7EuWvDWlzdXos8UmFiF/tp'
    'LfZjO3Yszel9P4k2iodWqBftMC+y29XF08U/S4UGHA=='
)
ORIGINAL_CASES_BYTES=zlib.decompress(base64.b64decode(_ORIGINAL_CASES_ZLIB_B64))
ORIGINAL_MANIFEST_BYTES=zlib.decompress(base64.b64decode(_ORIGINAL_MANIFEST_ZLIB_B64))
SOURCE_MANIFEST=json.loads(ORIGINAL_MANIFEST_BYTES)
ORIGINAL_CASES=json.loads(ORIGINAL_CASES_BYTES)
assert hashlib.sha256(ORIGINAL_CASES_BYTES).hexdigest()==SOURCE_SHA256_EXPECTED==SOURCE_MANIFEST['export_sha256_bytes']
assert len(ORIGINAL_CASES)==5 and [x['case_id'] for x in ORIGINAL_CASES]==CASE_IDS
for source in ORIGINAL_CASES:
    source_hash=hashlib.sha256(json.dumps(source,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()
    assert source_hash==SOURCE_CASE_SHA256_EXPECTED[source['case_id']]

# Keep originals precisely unchanged on disk; do NOT pass researcher-only metadata or IDs to the LLM.
for path,original_bytes in [(OUT/'original_cases_unmodified.json',ORIGINAL_CASES_BYTES),
                            (OUT/'original_export_manifest_unmodified.json',ORIGINAL_MANIFEST_BYTES)]:
    if path.exists(): assert path.read_bytes()==original_bytes, 'Original source mismatch in existing run.'
    else: path.write_bytes(original_bytes)

SOURCE_METRIC_MAP={'heart_rate_bpm':'hr_bpm','rmssd_ms':'rmssd_ms','eda_microsiemens':'eda_us'}
SOURCE_GOAL_MAP={'REST_SLEEP':'sleep','WAKEFULNESS':'awake','UNSPECIFIED':'unspecified'}
SOURCE_ACTIVITY_MAP={'RESTING':'RESTING','MOVEMENT':'MOVING','UNKNOWN':'UNKNOWN'}
SOURCE_QUALITY_MAP={'GOOD':'GOOD','MISSING':'MISSING','UNRELIABLE':'POOR','POOR':'POOR','UNKNOWN':'UNKNOWN'}
SOURCE_DESCRIPTION={'PILOT_201':'Ordinary evening comfort','PILOT_202':'Unknown preferences',
 'PILOT_203':'Unreliable/missing comparisons','PILOT_204':'Reported recent movement',
 'PILOT_205':'Lighting locked; no procedure underway'}

def adapt_original(source):
    o=source['inputs']; prefs=o['patient_preferences']; staff=o['staff_restrictions'];ctx=o['context']
    physiology={}
    for src,dst in SOURCE_METRIC_MAP.items():
        x=o['physiology'][src]
        ref_stat=x['baseline_quality']['status']; cur_stat=x['current_quality']['status']
        assert ref_stat in SOURCE_QUALITY_MAP and cur_stat in SOURCE_QUALITY_MAP
        assert x['current_window_minutes']==5 and x['baseline_window_minutes']==5
        assert x['baseline_gap_minutes']==0
        quality_ok=ref_stat=='GOOD' and cur_stat=='GOOD'
        physiology[dst]={
          'reference':x['baseline'],'current':x['current'],'unit':METRICS[dst],
          'reference_quality':SOURCE_QUALITY_MAP[ref_stat],
          'current_quality':SOURCE_QUALITY_MAP[cur_stat],
          'reference_window_minutes':x['baseline_window_minutes'],
          'current_window_minutes':x['current_window_minutes'],
          'comparable':bool(quality_ok and x['baseline'] is not None and x['current'] is not None),
          # Preserve the exact original labels and source-quality explanations for model context.
          'source_quality':{'reference_status':ref_stat,'current_status':cur_stat,
              'reference_reason':x['baseline_quality']['reason'],'current_reason':x['current_quality']['reason'],
              'summary_method':x['summary_method'],'baseline_gap_minutes':x['baseline_gap_minutes']}
        }
    sounds=prefs['preferred_sound_types']
    assert sounds is None or isinstance(sounds,list) and len(sounds)<=1, 'Cannot collapse multiple preferred sounds.'
    assert prefs['sound_opt_out'] in [True,False,None]
    assert SOURCE_GOAL_MAP[ctx['environment_goal']] in ['sleep','awake','unspecified']
    assert SOURCE_ACTIVITY_MAP[ctx['recent_activity']['reported']] in ['MOVING','RESTING','UNKNOWN']
    assert staff['sound_allowed'] and 'NONE_SILENCE' in staff['allowed_sound_types'], 'Unanticipated source permissions; audit needed.'
    p={'lighting_changes_allowed':prefs['automatic_lighting_changes_allowed'],
       # Unknown opt-out is NOT positive consent, but it is not a patient prohibition in this advisory prototype.
       'sound_changes_allowed':prefs['sound_opt_out'] is not True,
       'sound_opt_out_original':prefs['sound_opt_out'],
       'preferred_sound':sounds[0] if sounds else None,
       'preferred_tones':prefs['preferred_light_tones'],
       'prohibited_tones':prefs['prohibited_light_tones'],
       'prohibited_sounds':prefs['prohibited_sound_types']}
    s={'lighting_locked':staff['lighting_locked'],
       'brightness_min':staff['brightness_percent']['min'],
       'brightness_max':staff['brightness_percent']['max'],
       'allowed_tones':staff['allowed_light_tones'],
       'sound_allowed':staff['sound_allowed'],
       'allowed_sounds':[v for v in staff['allowed_sound_types'] if v!='NONE_SILENCE'],
       'max_volume':staff['max_volume_percent'],
       'max_duration_minutes':staff['max_duration_minutes'],
       # No original case establishes an extra blanket no-sound-during-procedure rule.
       'no_sound_during_procedure':False}
    r={'brightness':o['room']['lighting']['brightness_percent'],
       'tone':o['room']['lighting']['tone'],
       'sound':o['room']['sound']['type'],
       'volume':o['room']['sound']['volume_percent'],
       'remaining_minutes':o['room']['sound']['remaining_duration_minutes']}
    c={'case_id':source['case_id'],'phase':'development','title':SOURCE_DESCRIPTION[source['case_id']],
       'provenance':'WESAD-derived recorded physiology; hypothetical patient, hospital, room and staff context.',
       'physiology':physiology,
       'context':{'local_time':ctx['local_time'],'goal':SOURCE_GOAL_MAP[ctx['environment_goal']],
          'goal_original':ctx['environment_goal'],
          'reported_activity':SOURCE_ACTIVITY_MAP[ctx['recent_activity']['reported']],
          'reported_activity_original':ctx['recent_activity']['reported'],
          'minutes_since_activity':ctx['recent_activity']['minutes_since_activity'],
          'activity_note':ctx['recent_activity']['note'],
          'source_activity':ctx['source_activity'],
          'procedure':ctx['care_context']['procedure_in_progress'],
          'care_note':ctx['care_context']['note']},
       'room':r,'preferences':p,'staff':s}
    return c

CASES=[adapt_original(source) for source in ORIGINAL_CASES]
assert CASES_JSON_PATH is None,'Do not override the original cohort.'

def check_case(c):
    assert c['phase']=='development'
    assert re.fullmatch(r'PILOT_20[1-5]',c['case_id'])
    assert c['provenance'] and c['context']['goal'] in ['sleep','awake','unspecified']
    assert re.fullmatch(r'(?:[01]\d|2[0-3]):[0-5]\d',c['context']['local_time'])
    assert c['context']['reported_activity'] in ['RESTING','MOVING','UNKNOWN']
    assert set(c['physiology'])==set(METRICS)
    for k in METRICS:
        x=c['physiology'][k]
        assert x['unit']==METRICS[k]
        for role in ['reference','current']:
            v=x[role]
            assert v is None or type(v) in [int,float] and math.isfinite(v) and v>=0
            assert x[role+'_quality'] in ['GOOD','POOR','MISSING','UNKNOWN']
            assert x[role+'_window_minutes']==5
        assert type(x['comparable']) is bool
    s=c['staff'];r=c['room'];p=c['preferences']
    for v in [s['brightness_min'],s['brightness_max'],s['max_volume'],s['max_duration_minutes'],r['brightness'],r['volume'],r['remaining_minutes']]:
        assert type(v) in [int,float] and math.isfinite(v)
    assert 0<=s['brightness_min']<=r['brightness']<=s['brightness_max']<=100
    assert r['tone'] in s['allowed_tones'] and set(s['allowed_tones'])<=set(TONES)
    assert r['sound'] in SOUNDS and r['sound']!='NO_CHANGE'
    assert 0<=r['volume']<=100 and r['remaining_minutes']>=0
    if r['sound']=='NONE_SILENCE': assert r['volume']==0 and r['remaining_minutes']==0
    assert 0<=s['max_volume']<=100 and s['max_duration_minutes']>=0
    assert set(s['allowed_sounds'])<=set(SOUNDS)-{'NO_CHANGE','NONE_SILENCE'}
    assert p['preferred_sound'] is None or p['preferred_sound'] in SOUNDS[:-1]
    assert p['preferred_tones'] is None or set(p['preferred_tones'])<=set(TONES)
    assert set(p['prohibited_tones'])<=set(TONES)
    assert set(p['prohibited_sounds'])<=set(SOUNDS)-{'NO_CHANGE'}
    assert p['sound_opt_out_original'] in [None,True,False]
    assert all(type(v) is bool for v in [p['lighting_changes_allowed'],p['sound_changes_allowed'],
          s['lighting_locked'],s['sound_allowed'],s['no_sound_during_procedure'],c['context']['procedure']])
for c in CASES:check_case(c)
assert len({c['case_id'] for c in CASES})==5

# Strict mapping audit against original fields, including the deliberate missing data and patient/staff input states.
for source,c in zip(ORIGINAL_CASES,CASES):
    o=source['inputs'];assert source['case_id']==c['case_id']
    for a,b in SOURCE_METRIC_MAP.items():
        assert o['physiology'][a]['baseline']==c['physiology'][b]['reference']
        assert o['physiology'][a]['current']==c['physiology'][b]['current']
        assert o['physiology'][a]['baseline_quality']['status']==c['physiology'][b]['source_quality']['reference_status']
        assert o['physiology'][a]['current_quality']['status']==c['physiology'][b]['source_quality']['current_status']
    assert c['context']['goal_original']==o['context']['environment_goal']
    assert c['context']['reported_activity_original']==o['context']['recent_activity']['reported']
    assert c['context']['procedure']==o['context']['care_context']['procedure_in_progress']
    assert c['room']['brightness']==o['room']['lighting']['brightness_percent']
    assert c['room']['tone']==o['room']['lighting']['tone']
    assert c['room']['sound']==o['room']['sound']['type']
    assert c['room']['volume']==o['room']['sound']['volume_percent']
    assert c['room']['remaining_minutes']==o['room']['sound']['remaining_duration_minutes']
    assert c['staff']['lighting_locked']==o['staff_restrictions']['lighting_locked']
    assert c['staff']['allowed_tones']==o['staff_restrictions']['allowed_light_tones']
    assert sorted(c['staff']['allowed_sounds']+['NONE_SILENCE'])==sorted(o['staff_restrictions']['allowed_sound_types'])
    assert c['preferences']['sound_opt_out_original']==o['patient_preferences']['sound_opt_out']
    assert c['preferences']['preferred_tones']==o['patient_preferences']['preferred_light_tones']
    assert c['preferences']['prohibited_tones']==o['patient_preferences']['prohibited_light_tones']

assert all(not c['context']['procedure'] for c in CASES)
assert CASES[2]['physiology']['eda_us']['current'] is None
assert all(comparison['current'] is None or comparison['current']>=0 for c in CASES for comparison in c['physiology'].values())
assert CASES[3]['context']['reported_activity']=='MOVING'
assert CASES[4]['staff']['lighting_locked'] is True
assert CASES[4]['context']['procedure'] is False
write_frozen(OUT/'development_cases.json',CASES)
write_frozen(OUT/'normalization_audit.json',{
 'original_byte_sha256':SOURCE_SHA256_EXPECTED,'original_case_sha256_canonical':SOURCE_CASE_SHA256_EXPECTED,
 'source_to_normalized_metric_keys':SOURCE_METRIC_MAP,'goal_enum_map':SOURCE_GOAL_MAP,
 'activity_enum_map':SOURCE_ACTIVITY_MAP,'quality_enum_map':SOURCE_QUALITY_MAP,
 'note':'Original case bytes untouched and bundled; model sees faithful task data, not researcher-only provenance. Source UNRELIABLE becomes normalized POOR; raw quality status and reason retained. UNKNOWN opt-out remains explicit in preferences. No extra procedure prohibition invented.',
 'completed_preflight':'OFFLINE_ONLY'})
display(pd.DataFrame([{'case':c['case_id'],'scenario':c['title'],'goal':c['context']['goal'],
 'reported_activity':c['context']['reported_activity'],'locked_lighting':c['staff']['lighting_locked'],
 'unavailable_metrics':','.join(k for k,v in c['physiology'].items() if not v['comparable'])} for c in CASES]))
print('Source bytes and all five canonical hashes verified; original cases preserved. No API calls.')


In [ ]:
# 6 · Four bounded tools: no arbitrary code execution

def comparisons(c):
    out={}
    for k,p in c['physiology'].items():
        usable=(p['reference'] is not None and p['current'] is not None and
                p['reference_quality']==p['current_quality']=='GOOD' and p['comparable'])
        delta=p['current']-p['reference'] if usable else None
        pct=100*delta/p['reference'] if usable and p['reference']!=0 and k!='eda_us' else None
        out[k]={'status':'AVAILABLE' if usable else 'UNAVAILABLE',
                'absolute_change':None if delta is None else round(delta,6),
                'percent_change':None if pct is None else round(pct,6)}
    return out

def tool(name, args, c):
    if name=='calculate_changes': return comparisons(c)
    if name=='check_data_quality':
        return {k:{**{f:p[f] for f in ['reference_quality','current_quality','reference_window_minutes','current_window_minutes','comparable']},
                   'comparison_available':comparisons(c)[k]['status']=='AVAILABLE'} for k,p in c['physiology'].items()}
    if name=='check_activity_context': return copy.deepcopy(c['context'])
    if name=='retrieve_evidence':
        ids=args['evidence_ids']; known={e['id'] for e in EVIDENCE}
        if not set(ids)<=known: raise ValueError('Unknown evidence ID')
        return [e for e in EVIDENCE if e['id'] in ids]
    raise ValueError('Undeclared tool: '+name)

def spec(name, description, properties=None, required=None):
    return {'type':'function','function':{'name':name,'description':description,
        'parameters':{'type':'object','properties':properties or {},'required':required or [],'additionalProperties':False}}}
TOOLS=[spec('calculate_changes','Calculate descriptive HR, RMSSD and EDA changes from the supplied case; no arguments.'),
       spec('check_data_quality','Inspect recorded quality and comparability, without diagnosing signal validity.'),
       spec('check_activity_context','Return patient context and separate source activity exactly as supplied.'),
       spec('retrieve_evidence','Retrieve complete research summaries by evidence_ids before using their findings or citing them. The index gives identifiers, not findings. Returns roles, applicable actions and limitations; it does not retrieve full papers.',
            {'evidence_ids':{'type':'array','items':{'type':'string','enum':[e['id'] for e in EVIDENCE]},'minItems':1}},['evidence_ids'])]
TOOL_SCHEMAS={x['function']['name']:x['function']['parameters'] for x in TOOLS}

In [ ]:
# 7 · Output contract and deterministic simulation

def obj(props): return {'type':'object','properties':props,'required':list(props),'additionalProperties':False}
def enum(values): return {'type':'string','enum':values}
def text_schema(): return {'type':'string','minLength':1,'pattern':r'\S'}
RESEARCH_IDS={e['id'] for e in EVIDENCE if e['kind']!='prototype_rule'}
POLICY_IDS={e['id'] for e in EVIDENCE if e['kind']=='prototype_rule'}
def ids_schema(ids): return {'type':'array','uniqueItems':True,'items':enum(sorted(ids))}
COMPARISON_SCHEMA=obj({'status':enum(['AVAILABLE','UNAVAILABLE']),
                      'absolute_change':{'type':['number','null']},'percent_change':{'type':['number','null']}})
OUTPUT_SCHEMA=obj({
 'physiological_interpretation':text_schema(), 'physiology_evidence_ids':ids_schema(RESEARCH_IDS), 'certainty':enum(['LOW','MEDIUM','HIGH']), 'certainty_rationale':text_schema(),
 'activity_assessment':enum(['RESTING','MOVING','UNKNOWN']),
 'comparisons':obj({k:COMPARISON_SCHEMA for k in METRICS}),
 'lighting':obj({'action':enum(LIGHT_ACTIONS),'tone':enum(TONES),'rationale':text_schema(),'evidence_ids':ids_schema(RESEARCH_IDS),'policy_ids':ids_schema(POLICY_IDS)}),
 'sound':obj({'option':enum(SOUNDS),'rationale':text_schema(),'evidence_ids':ids_schema(RESEARCH_IDS),'policy_ids':ids_schema(POLICY_IDS)}),
 'limitations':{'type':'array','items':text_schema(),'minItems':1}})

def settings_for(answer,c):
    r=c['room']; s=c['staff']; action=answer['lighting']['action']
    direction={'INCREASE':1,'DECREASE':-1,'NO_CHANGE':0}[action]
    # This is the declared action-to-setting mapping, not repair of invalid model output.
    brightness=min(s['brightness_max'],max(s['brightness_min'],r['brightness']+direction*POLICY['brightness_step']))
    option=answer['sound']['option']
    if option=='NO_CHANGE': sound,vol,dur=r['sound'],r['volume'],r['remaining_minutes']
    elif option=='NONE_SILENCE': sound,vol,dur=option,0,0
    else: sound,vol,dur=option,min(POLICY['sound_volume'],s['max_volume']),min(POLICY['sound_duration_minutes'],s['max_duration_minutes'])
    return {'brightness':brightness,'tone':answer['lighting']['tone'],'sound':sound,'volume':vol,'duration_minutes':dur}

def validate_answer(a,c,available_research_ids=None):
    errors=[f'SCHEMA {list(e.path)}: {e.message}' for e in Draft202012Validator(OUTPUT_SCHEMA).iter_errors(a)]
    if errors: return errors,None
    # Fail closed unless research availability is explicitly supplied by the workflow.
    available=set(available_research_ids or [])
    cited=set(a['physiology_evidence_ids'])
    for field in ['lighting','sound']: cited.update(a[field]['evidence_ids'])
    # Also catch IDs named inside prose, not just the structured citation lists.
    prose=[a['certainty_rationale'],a['physiological_interpretation'],a['lighting']['rationale'],a['sound']['rationale'],*a['limitations']]
    inline=set(re.findall(r'\bE[0-9]+\b',' '.join(prose)))
    for eid in sorted(inline-RESEARCH_IDS-POLICY_IDS): errors.append('UNKNOWN_CITATION '+eid)
    cited.update(inline & RESEARCH_IDS)
    for eid in sorted(cited-available): errors.append('RESEARCH_NOT_AVAILABLE '+eid)
    for eid in a['physiology_evidence_ids']:
        entry=next(e for e in EVIDENCE if e['id']==eid)
        if 'physiology' not in entry['context_fields']: errors.append('EVIDENCE_FIELD_MISMATCH physiology/'+eid)
    truth=comparisons(c)
    for k in METRICS:
        x=a['comparisons'][k]; expected=truth[k]
        if x['status']!=expected['status']: errors.append('COMPARISON_STATUS '+k)
        for f in ['absolute_change','percent_change']:
            v=x[f]; t=expected[f]
            if (t is None and v is not None) or (t is not None and (v is None or not math.isfinite(v) or abs(v-t)>POLICY['comparison_tolerance'])):
                errors.append('COMPARISON_VALUE '+k+'/'+f)
    if a['activity_assessment']!=c['context']['reported_activity']: errors.append('ACTIVITY_MISMATCH')
    cfg=settings_for(a,c); s=c['staff']; p=c['preferences']; r=c['room']; l=a['lighting']; snd=a['sound']['option']
    if s['lighting_locked'] or not p['lighting_changes_allowed']:
        if l['action']!='NO_CHANGE' or l['tone']!=r['tone']: errors.append('LIGHTING_LOCK')
    if l['tone'] not in s['allowed_tones']: errors.append('TONE_RESTRICTION')
    if l['tone'] in p.get('prohibited_tones',[]): errors.append('PATIENT_TONE_PROHIBITION')
    if l['action']=='INCREASE' and cfg['brightness']<=r['brightness']: errors.append('NO_INCREASE_POSSIBLE')
    if l['action']=='DECREASE' and cfg['brightness']>=r['brightness']: errors.append('NO_DECREASE_POSSIBLE')
    active=cfg['sound']!='NONE_SILENCE'
    forbidden=not s['sound_allowed'] or (c['context']['procedure'] and s['no_sound_during_procedure'])
    if active and forbidden: errors.append('SOUND_RESTRICTION')
    if active and cfg['sound'] not in s['allowed_sounds']: errors.append('SOUND_NOT_ALLOWED')
    if cfg['sound'] in p['prohibited_sounds']: errors.append('PATIENT_SOUND_PROHIBITION')
    if active and (cfg['volume']<=0 or cfg['duration_minutes']<=0): errors.append('SOUND_WITHOUT_PLAYBACK')
    if cfg['volume']>s['max_volume'] or cfg['duration_minutes']>s['max_duration_minutes']: errors.append('PLAYBACK_LIMIT')
    changed=(cfg['sound'],cfg['volume'],cfg['duration_minutes'])!=(r['sound'],r['volume'],r['remaining_minutes'])
    if changed and not p['sound_changes_allowed']: errors.append('SOUND_CHANGE_PROHIBITED')
    if snd in ['PREFERRED_CALMING_MUSIC','WHITE_NOISE'] and p['preferred_sound']!=snd:
        errors.append('PREFERENCE_REQUIRED_FOR_THIS_PROTOTYPE_SOUND')
    night=int(c['context']['local_time'][:2])>=POLICY['night_start'] or int(c['context']['local_time'][:2])<POLICY['night_end']
    for field,choice,allowed_key in [('lighting',l['action'],'lighting_actions'),('sound',snd,'sounds')]:
        applicable=False
        for eid in a[field]['evidence_ids']+a[field]['policy_ids']:
            e=next(e for e in EVIDENCE if e['id']==eid)
            if e.get('role')=='context_only':
                if field not in e['context_fields']: errors.append('EVIDENCE_FIELD_MISMATCH '+field+'/'+eid)
                continue  # contextual citation is permitted but cannot supply action support
            if choice not in e[allowed_key]:
                errors.append(('POLICY_ACTION_MISMATCH ' if eid in POLICY_IDS else 'EVIDENCE_ACTION_MISMATCH ')+field+'/'+eid); continue
            condition_ok=True
            if e['condition']=='time_of_day':
                condition_ok=(choice=='NO_CHANGE' or (choice=='DECREASE' and night) or
                              (choice=='INCREASE' and not night and c['context']['goal']!='sleep'))
            if e['condition']=='sound_permission': condition_ok=not forbidden
            if e['condition']=='sleep_goal': condition_ok=not forbidden and c['context']['goal'] in ['sleep','rest']
            if e['condition']=='preference_compatible' and choice in ['PREFERRED_CALMING_MUSIC','NATURE_SOUNDS','WHITE_NOISE']:
                condition_ok=p['preferred_sound']==choice
            if not condition_ok: errors.append('EVIDENCE_CONTEXT_MISMATCH '+field+'/'+eid)
            applicable=applicable or condition_ok
        if not applicable: errors.append('NO_APPLICABLE_SUPPORT '+field)
    return errors, cfg if not errors else None

CITATION_ERROR_PREFIXES=('UNKNOWN_CITATION ', 'RESEARCH_NOT_AVAILABLE ',
    'EVIDENCE_FIELD_MISMATCH ', 'POLICY_ACTION_MISMATCH ', 'EVIDENCE_ACTION_MISMATCH ',
    'EVIDENCE_CONTEXT_MISMATCH ', 'NO_APPLICABLE_SUPPORT ')

def assess_answer(answer,c,available_research_ids=None):
    errors,settings=validate_answer(answer,c,available_research_ids)
    schema_errors=[e for e in errors if e.startswith('SCHEMA ')]
    # Do not infer domain success from an output that could not be structurally checked.
    if schema_errors:
        return {'schema_valid':False,'calculation_valid':None,'action_compliant':None,
          'citation_contract_valid':None,'schema_errors':schema_errors,
          'calculation_errors':[],'action_errors':[],'citation_errors':[],
          'overall_valid':False,'errors':errors,'settings':None,'semantic_grounding':'NOT_ASSESSED'}
    citation=[e for e in errors if e.startswith(CITATION_ERROR_PREFIXES)]
    calculation=[e for e in errors if e.startswith(('COMPARISON_STATUS ','COMPARISON_VALUE '))]
    action=[e for e in errors if e not in citation+calculation]
    return {'schema_valid':True,'calculation_valid':not calculation,'action_compliant':not action,
      'citation_contract_valid':not citation,'schema_errors':[],
      'calculation_errors':calculation,'action_errors':action,'citation_errors':citation,
      'overall_valid':not errors,'errors':errors,'settings':settings,'semantic_grounding':'NOT_ASSESSED'}


## A compact prompt, with the scientific limits made explicit
The baseline receives the full fixed evidence table. The agent receives an index and can retrieve the same entries. Both use the same output schema and staff/patient rules. This tests the **whole agent workflow**, including tool access and one revision; it does not isolate reasoning alone. The comparison formula tolerance is a development setting, not a clinical threshold.

In [ ]:
# 8 · Shared instructions and condition-specific evidence access
COMMON = """You are a non-clinical AHR research prototype. Recommend simulated lighting and sound for staff review.
Use only supplied case facts and evidence. Never diagnose stress or claim a patient is relaxed from these signals alone.
Reported activity describes what the patient is doing; it is not a physiological diagnosis. Measurement changes neither confirm nor contradict RESTING by themselves. Mention recent movement as relevant context, not a proven cause or a way to exclude pathology.
Do not classify readings as normal/abnormal, infer a clinical need, or rule out disease from these case comparisons. Missing EDA removes that comparison; available EDA would still not establish a complete autonomic or psychological assessment.
Prototype rules permit actions; they do not establish a benefit or impose restrictions absent from the actual case. A research-based claim needs support even if you omit its citation. If that support is unavailable, use a descriptive statement or explicitly acknowledge the uncertainty.
Respect explicit staff restrictions and patient prohibitions. UNKNOWN stays unknown. An UNSPECIFIED environment goal stays unspecified; do not infer a comfort or sleep goal. Distinguish reported hospital activity from source activity. The original 'MOVEMENT' activity label is normalized to 'MOVING'; the original label remains present. An original UNRELIABLE quality label is normalized to POOR for comparability, with source status/reasons preserved.
Consider time of day, room settings, preferences, activity and data quality. No change or silence may be appropriate; explain that choice using the case context, especially when an explicit permitted preference is supplied. Do not merely state that an option is allowed.
Brightness/volume/duration are assigned by the supplied prototype policy; do not generate those numbers.
Return a JSON object exactly matching OUTPUT_SCHEMA, in assistant content, without markdown or extra fields.
Lighting action describes brightness direction; tone is a separate choice. A lighting lock also locks tone.
Provide brief evidence-grounded rationales and limitations, not private step-by-step reasoning.
Comparisons: signed absolute=current-reference. HR/RMSSD percent=100*(current-reference)/reference, null if zero denominator.
For EDA report absolute change only and null percent. Missing, POOR/UNKNOWN quality or noncomparable pairs require UNAVAILABLE and null changes.
Calculate at full input precision, round final values to 3 decimal places; tolerance is 0.01 in the reported unit/percentage points.
activity_assessment must copy context.reported_activity. staff.sound_allowed is explicit staff permission; preferences do not override staff restrictions.
A research finding does not prove this case will benefit. Do not give universal safety assurances or invent quantitative uncertainty.
Patient preferred light tones are preferences, not mandatory restrictions; prohibited tones are mandatory. A null original sound_opt_out is unknown, NOT positive consent; this is an advisory simulated recommendation with no automatic actuation. Unknown sound preference does not mean staff prohibition. Music/white-noise choices require that explicit preference in this prototype.
Use evidence_ids only for research entries and policy_ids only for supplied prototype rules, as enumerated in OUTPUT_SCHEMA.
Certainty means your qualitative confidence that the whole recommendation fits the supplied case, not confidence in a diagnosis, causal benefit, or arithmetic. It is not a calibrated probability.
Use LOW when important context/evidence is missing or conflicting; MEDIUM when a reasonable contextual choice exists but alternatives and transfer uncertainty remain; HIGH only when explicit recorded facts/constraints make the whole choice unusually clear. Good signal quality, correct calculations, retrieved citations or a validation pass alone do not justify HIGH.
Explain the label in certainty_rationale using specific known facts and remaining uncertainty. Mechanical validation does not judge whether the confidence is justified; that requires researcher review.
A permitted action is not proof of effectiveness. Do not assert that the current environment supports rest unless that response is actually recorded.
List research used for physiological interpretation in physiology_evidence_ids; use [] for descriptive case facts alone.
Every lighting/sound choice needs an applicable action_support source or optional_action_basis. Cite only bases that actually cover the chosen action; omit an inapplicable policy ID even when a research source supports the action. Prototype preference support applies equally to preferred nature sounds, music and white noise. A context_only source may support a relevant caveat but cannot alone justify an action.
Check each entry's declared role, field, action and condition. Explicit preference and permission constraints still apply to research-supported choices.
Empty research lists are allowed for actions supported by supplied prototype rules. These rules establish permitted behavior, not efficacy.
Cite research only after its complete TABLE ENTRY has been supplied or retrieved; this does not mean you have read the full paper.
Separate study findings from your contextual inference and prototype settings. Preserve null findings, study populations, exposure durations and stated uncertainty.
If support is absent from the supplied material, say so narrowly; do not claim that no evidence exists anywhere.
The mandatory_constraints object states requirements; optional_action_bases states possible rationales, not requirements. These categories have different force.
Select an action from the output schema within the case restrictions, then justify it with applicable support. Do not intersect all sources' supported-action lists: they describe alternative justifications, not cumulative prohibitions.
The entries in optional_action_bases are optional. They cannot override, lock or veto a research-supported alternative. policy_ids is a legacy output field for these IDs, not a declaration that their actions are mandatory.
"""

def messages_for(c,arm):
    extra=('You may call any declared tools as needed, in any order, or answer directly. Research index entries are discovery metadata only, not findings. Before using a research finding or citation anywhere in the answer, retrieve its complete entry using retrieve_evidence. Do not infer findings from titles or IDs. No research summaries are initially available to you. You can answer directly using supplied optional action bases where applicable. A final-only revision cannot retrieve missing evidence. Use native tool calls only for declared functions; JSON is a format, not a tool.'
           if arm=='agent' else 'Answer once without tools.')
    research=[e for e in EVIDENCE if e['id'] in RESEARCH_IDS]
    evidence=research if arm=='baseline' else [{k:e[k] for k in ['id','kind','topic','source','role']} for e in research]
    return [{'role':'system','content':COMMON+'\n'+extra},
            {'role':'user','content':encode({'case':c,'simulation_policy':POLICY,'mandatory_constraints':{'staff':copy.deepcopy(c['staff']),'patient_permissions_and_prohibitions':{k:copy.deepcopy(v) for k,v in c['preferences'].items() if k!='preferred_sound'},'procedure_active':c['context']['procedure']},'optional_action_bases':[e for e in EVIDENCE if e['id'] in ['E04','E05']],'calculation_conventions':[e for e in EVIDENCE if e['id']=='E06'],'OUTPUT_SCHEMA':OUTPUT_SCHEMA,
                                            'evidence_table' if arm=='baseline' else 'evidence_index':evidence})}]

## Connection and spending controls
All offline checks run **before** credential loading or generation. Before live execution, an endpoint metadata GET checks price, context/output capacity, required parameters and explicit `tool_choice=auto` support. It does not test live availability or model quality. Routing is restricted to the checked endpoint tags, with price ceilings of **$0.25/M input and $1.50/M output**. There is no model fallback.

Before every HTTP attempt, including a retry, the client records a cost reservation and checks the run budget, per-request ceiling and attempt count. Unknown cost remains reserved; timeouts are not retried automatically. A reported cost above its reservation stops further calls. Successful HTTP envelopes containing provider errors are classified as failures.

The input estimate is deliberately conservative, **not an exact tokenizer count or guaranteed bill**. The output cap includes reasoning. Key/account spending controls are separate. Request/response logs omit authorization headers and the key. Run-folder identity is checked before exporting inputs, and completed or interrupted workflows cannot silently rerun. Use one Colab runtime per result folder; the lock detects concurrent runs in that folder, not separate machines.

References: [Google Gemini 3 temperature](https://ai.google.dev/gemini-api/docs/gemini-3#temperature), [OpenRouter routing/prices](https://openrouter.ai/docs/guides/routing/provider-selection), [reasoning continuation](https://openrouter.ai/docs/guides/best-practices/reasoning-tokens).


In [ ]:
# 9 · Bounded transport; no network is used merely by defining this cell
class RunStopped(Exception): pass

HALT_STATUSES={'PROVIDER_FAILURE','GLOBAL_BUDGET_STOP','BUDGET_STOP','REQUEST_LIMIT','REQUEST_COST_STOP','INPUT_SIZE_STOP',
 'REPORTED_COST_EXCEEDED_LOCAL_ESTIMATE','UPSTREAM_IDENTITY_MISMATCH','EMPTY_CHOICES','MALFORMED_RESPONSE','RUNNING'}

def provider_error(raw,http_status):
    if not isinstance(raw,dict): return {'code':http_status,'message':'Non-object response','transient':False}
    choices=raw.get('choices'); choice=choices[0] if isinstance(choices,list) and choices and isinstance(choices[0],dict) else {}
    error=raw.get('error') or choice.get('error')
    if error is None and choice.get('finish_reason')!='error' and http_status==200: return None
    error=error if isinstance(error,dict) else {'message':str(error or 'Provider generation error')}
    try: code=int(error.get('code',http_status))
    except (TypeError,ValueError): code=http_status
    return {'code':code,'message':str(error.get('message','Provider error')),
            'transient':code in [429,500,502,503,504]}

def price(value):
    try: result=float(value)
    except (TypeError,ValueError): raise ValueError('Missing/invalid endpoint price')
    require(math.isfinite(result) and result>=0,'Invalid endpoint price')
    return result

def eligible_endpoints(data):
    require(isinstance(data,dict) and data.get('id')==MODEL,'Endpoint catalogue model mismatch.')
    approved=[]
    required={'tools','tool_choice','max_tokens'}
    if SEND_TEMPERATURE: required.add('temperature')
    if REASONING_EFFORT is not None: required.add('reasoning')
    for e in data.get('endpoints',[]):
        try:
            tag=e.get('tag'); require(isinstance(tag,str) and tag,'Missing endpoint tag')
            require(e.get('provider_name')==PROVIDER,'Provider mismatch')
            require(not any(x in tag.lower() for x in ['/flex','/priority','/fast']), 'Nonstandard provider route excluded')
            require(required<=set(e.get('supported_parameters') or []),'Missing parameter capability')
            require((e.get('supports_tool_choice') or {}).get('auto') is True,'Auto tool choice not explicitly supported')
            require(e.get('status')==0,'Endpoint not listed as enabled')
            require(isinstance(e.get('max_completion_tokens'),int) and e['max_completion_tokens']>=MAX_OUTPUT_TOKENS,'Output cap unsupported')
            require(isinstance(e.get('context_length'),int) and e['context_length']>=MAX_OUTPUT_TOKENS+MAX_ESTIMATED_INPUT_TOKENS,'Context allowance insufficient')
            prices=e['pricing']
            require(price(prices['prompt'])*1e6<=MAX_INPUT_PRICE_PER_M+1e-9,'Input price too high')
            require(price(prices['completion'])*1e6<=MAX_OUTPUT_PRICE_PER_M+1e-9,'Output price too high')
            require(price(prices.get('request',0))==0,'Per-request fee unsupported')
            require(price(prices.get('internal_reasoning',prices['completion']))*1e6<=MAX_OUTPUT_PRICE_PER_M+1e-9,'Reasoning price too high')
            approved.append(e)
        except (ValueError,KeyError,TypeError): continue
    require(approved,'No explicitly compatible endpoint within the configured prices/cap. No generation sent.')
    # Freeze one and only one endpoint. Do not allow fallback or cross-provider routing.
    approved.sort(key=lambda e:(price(e['pricing']['completion']),price(e['pricing']['prompt']),str(e['tag'])))
    return [approved[0]]

def master_used_usd():
    # Includes known charges and conservatively reserved unknown-cost requests.
    total=0.0
    for name in PROFILES:
        p=MASTER_ROOT/name/'cost_ledger.json'
        if not p.exists(): continue
        ledger=json.loads(p.read_text())
        require(isinstance(ledger,list),'Invalid cross-model cost ledger: '+name)
        for item in ledger:
            amount=item.get('accounted_usd') if isinstance(item,dict) else None
            require(type(amount) in (float,int) and math.isfinite(amount) and amount>=0,'Invalid cost entry: '+name)
            total+=amount
    return total

class Router:
    def __init__(self,key,out,endpoints=None):
        self.key=key;self.out=Path(out);self.out.mkdir(parents=True,exist_ok=True);self.last=0
        self.path=self.out/'cost_ledger.json';self.endpoints=endpoints or []
        self.reload()
    def reload(self):
        self.ledger=json.loads(self.path.read_text()) if self.path.exists() else []
        require(isinstance(self.ledger,list),'Corrupt ledger')
        for x in self.ledger:
            require(isinstance(x,dict) and type(x.get('accounted_usd')) in [int,float] and math.isfinite(x['accounted_usd']) and x['accounted_usd']>=0,'Corrupt ledger; stop for review')
    def get(self,path):
        r=requests.get('https://openrouter.ai/api/v1/'+path,headers={'Authorization':'Bearer '+self.key},timeout=30)
        r.raise_for_status();return r.json()
    def used(self): return sum(x['accounted_usd'] for x in self.ledger)
    def payload(self,messages,use_tools):
        require(self.endpoints,'Endpoint preflight must complete before generation.')
        require(len(self.endpoints)==1 and self.endpoints[0]['provider_name']==PROVIDER,'Exactly one verified provider endpoint must be pinned.')
        provider={'require_parameters':True,'max_price':{'prompt':MAX_INPUT_PRICE_PER_M,'completion':MAX_OUTPUT_PRICE_PER_M,'request':0},
                  'only':[self.endpoints[0]['tag']],'allow_fallbacks':False}
        payload={'model':MODEL,'messages':messages,'max_tokens':MAX_OUTPUT_TOKENS,'provider':provider}
        if SEND_TEMPERATURE: payload['temperature']=TEMPERATURE
        if REASONING_EFFORT is not None: payload['reasoning']={'effort':REASONING_EFFORT}
        if use_tools: payload.update(tools=TOOLS,tool_choice='auto')
        return payload
    def reservation(self,payload):
        estimate=2*len(encode(payload).encode('utf-8'))+4096
        return estimate,(estimate*MAX_INPUT_PRICE_PER_M+MAX_OUTPUT_TOKENS*MAX_OUTPUT_PRICE_PER_M)/1e6
    def call(self,messages,use_tools,tag):
        payload=self.payload(messages,use_tools);attempts=[]
        for retry in range(TRANSIENT_RETRIES+1):
            self.reload()
            estimate,reserve=self.reservation(payload)
            if len(self.ledger)>=MAX_HTTP_REQUESTS:return {'error':'REQUEST_LIMIT','attempts':attempts}
            if estimate>MAX_ESTIMATED_INPUT_TOKENS:return {'error':'INPUT_SIZE_STOP','attempts':attempts}
            if reserve>MAX_REQUEST_RESERVATION_USD+1e-12:return {'error':'REQUEST_COST_STOP','attempts':attempts}
            if self.used()+reserve>RUN_BUDGET_USD+1e-12:return {'error':'BUDGET_STOP','attempts':attempts}
            if master_used_usd()+reserve>GLOBAL_BUDGET_USD+1e-12:return {'error':'GLOBAL_BUDGET_STOP','attempts':attempts}
            wait=max(0,PACE_SECONDS-(time.monotonic()-self.last))
            if wait:time.sleep(wait)
            uid=uuid.uuid4().hex
            item={'id':uid,'tag':tag,'time':stamp(),'reserved_usd':reserve,'accounted_usd':reserve,'cost_known':False}
            self.ledger.append(item);save_json(self.path,self.ledger)
            save_json(self.out/'requests'/f'{uid}_request.json',payload)
            started=time.monotonic();self.last=started;r=None
            try:
                r=requests.post('https://openrouter.ai/api/v1/chat/completions',headers={'Authorization':'Bearer '+self.key,'Content-Type':'application/json'},json=payload,timeout=REQUEST_TIMEOUT)
                try: raw=r.json()
                except ValueError:raw={'unparsed_body':r.text[:20000]}
                status=r.status_code
            except requests.RequestException as exc:
                raw={'transport_error':type(exc).__name__};status=0
            if not isinstance(raw,dict):raw={'malformed_body':raw}
            failure=provider_error(raw,status)
            attempt={'provider_error':failure,'id':uid,'http_status':status,'latency_seconds':time.monotonic()-started,'raw':raw}
            attempts.append(attempt);save_json(self.out/'requests'/f'{uid}_response.json',attempt)
            usage=raw.get('usage') if isinstance(raw.get('usage'),dict) else {};cost=usage.get('cost')
            if type(cost) in [int,float] and math.isfinite(cost) and cost>=0:item.update(accounted_usd=float(cost),cost_known=True)
            item.update(http_status=status,provider=raw.get('provider'),generation_id=raw.get('id'));save_json(self.path,self.ledger)
            if item['accounted_usd']>reserve+1e-12 or self.used()>RUN_BUDGET_USD+1e-12 or master_used_usd()>GLOBAL_BUDGET_USD+1e-12:
                return {'error':'REPORTED_COST_EXCEEDED_LOCAL_ESTIMATE','attempts':attempts}
            if failure is None:
                choices=raw.get('choices')
                if not isinstance(choices,list) or not choices:return {'error':'EMPTY_CHOICES','attempts':attempts}
                choice=choices[0]
                if not isinstance(choice,dict) or not isinstance(choice.get('message'),dict):return {'error':'MALFORMED_RESPONSE','attempts':attempts}
                if raw.get('model')!=MODEL or raw.get('provider') not in {e['provider_name'] for e in self.endpoints}:
                    return {'error':'UPSTREAM_IDENTITY_MISMATCH','attempts':attempts}
                return {'choice':choice,'raw':raw,'attempts':attempts}
            if failure['transient'] and retry<TRANSIENT_RETRIES:
                try:delay=float(r.headers.get('Retry-After','15')) if r is not None else 15
                except (ValueError,TypeError):delay=15
                if not math.isfinite(delay) or delay>60:break
                time.sleep(max(1,delay));continue
            break
        return {'error':'PROVIDER_FAILURE','attempts':attempts}

CLIENT=None

def prepare_client():
    require(globals().get('OFFLINE_CHECKS_PASSED') is True,'Run offline checks before live execution.')
    try:
        from google.colab import userdata
        key=userdata.get('OPENROUTER_API_KEY')
    except Exception:
        from getpass import getpass
        key=os.environ.get('OPENROUTER_API_KEY') or getpass('OpenRouter API key (hidden): ')
    require(isinstance(key,str) and bool(key.strip()),'An OpenRouter key is required.')
    client=Router(key,OUT)
    data=client.get('models/'+MODEL+'/endpoints')['data']
    client.endpoints=eligible_endpoints(data)
    save_json(OUT/('endpoint_preflight_'+uuid.uuid4().hex+'.json'),{'checked':stamp(),'data':data,'approved_tags':[e['tag'] for e in client.endpoints]})
    print('SINGLE selected provider:',PROVIDER,'| verified endpoint:',[e['tag'] for e in client.endpoints])
    print('HQ-based provider filter only; data residency is NOT independently verified.')
    print('Local limits: $',RUN_BUDGET_USD,'per run; $',MAX_REQUEST_RESERVATION_USD,'reservation per request;',MAX_HTTP_REQUESTS,'HTTP attempts.')
    print('These are local safeguards, not guaranteed account billing caps. Unknown cost stays reserved.')
    return client
print('Client defined. No network or generation calls in this cell.')


In [ ]:
# 10 · One reusable workflow for both conditions

def parse_final(content):
    if not isinstance(content,str) or not content.strip(): raise ValueError('EMPTY_FINAL_CONTENT')
    # No repair or markdown unwrapping: score the exact returned JSON contract.
    def unique(pairs):
        result={}
        for key,value in pairs:
            if key in result: raise ValueError('DUPLICATE_JSON_KEY '+key)
            result[key]=value
        return result
    return json.loads(content,object_pairs_hook=unique,parse_constant=lambda x: (_ for _ in ()).throw(ValueError('NONFINITE_JSON')))

def run_workflow(c,arm,client,call_limit=None):
    started=time.monotonic(); messages=messages_for(c,arm)
    result={'case_id':c['case_id'],'arm':arm,'model':MODEL,'started':stamp(),
        'status':'RUNNING','first_valid':None,'final_valid':False,'revision_used':False,
        'attempts':[],'tools':[],'final_attempts':[],'accepted':None,'settings':None}
    require(arm in ['baseline','agent'],'Invalid workflow arm')
    maxcalls=min(call_limit,AGENT_MAX_CALLS if arm=='agent' else 1) if call_limit is not None else (AGENT_MAX_CALLS if arm=='agent' else 1)
    require(type(maxcalls) is int and maxcalls>0,'Invalid call limit')
    revised=False; ntools=0; seen_ids=set()
    available_research=set(RESEARCH_IDS) if arm=='baseline' else set()
    for turn in range(maxcalls):
        use_tools=arm=='agent' and not revised
        response=client.call(messages,use_tools,f"{c['case_id']}/{arm}/{turn+1}")
        result['attempts'].extend(response.get('attempts',[]))
        if response.get('error'):
            result['status']=response['error']; break
        choice=response['choice']; message=choice.get('message') or {}
        if choice.get('finish_reason')=='length': result['status']='OUTPUT_TRUNCATED'; break
        if choice.get('finish_reason') in ['error','content_filter']:
            result['status']='GENERATION_'+choice['finish_reason'].upper(); break
        if not isinstance(message,dict): result['status']='MALFORMED_RESPONSE';break
        calls=message.get('tool_calls') or []
        if not isinstance(calls,list):result['status']='INVALID_TOOL_CALL';break
        if calls:
            if not use_tools: result['status']='UNEXPECTED_TOOL_CALL'; break
            if ntools+len(calls)>AGENT_MAX_TOOLS: result['status']='TOOL_LIMIT'; break
            # Retain reasoning details/signatures unchanged for continuation, not displayed as rationale.
            assistant={k:copy.deepcopy(message[k]) for k in ['role','content','tool_calls','reasoning_details','reasoning'] if k in message}
            assistant['role']='assistant'; messages.append(assistant)
            failed=False
            for tc in calls:
                name=None
                try:
                    require(isinstance(tc,dict) and tc.get('type')=='function' and isinstance(tc.get('function'),dict),'Malformed native tool call')
                    name=tc['function'].get('name');rawargs=tc['function'].get('arguments')
                    if not tc.get('id') or tc['id'] in seen_ids: raise ValueError('Missing/duplicate tool ID')
                    seen_ids.add(tc['id'])
                    if name not in TOOL_SCHEMAS: raise ValueError('Undeclared tool')
                    args=parse_final(rawargs) if isinstance(rawargs,str) else rawargs
                    Draft202012Validator(TOOL_SCHEMAS[name]).validate(args)
                    output=tool(name,args,c)
                    if name=='retrieve_evidence':
                        available_research.update(e['id'] for e in output if e['id'] in RESEARCH_IDS)
                    result['tools'].append({'name':name,'args':args,'output':output,'success':True})
                    ntools+=1
                    messages.append({'role':'tool','tool_call_id':tc['id'],'content':encode(output)})
                except Exception as exc:
                    result['tools'].append({'name':name,'success':False,'error':str(exc)})
                    result['status']='INVALID_TOOL_CALL'; failed=True; break
            if failed: break
            continue
        try:
            answer=parse_final(message.get('content')); assessment=assess_answer(answer,c,available_research)
            errors,settings=assessment['errors'],assessment['settings']
        except Exception as exc:
            answer=None; settings=None; errors=[str(exc)]
            assessment={'schema_valid':False,'calculation_valid':None,'action_compliant':None,
                'citation_contract_valid':None,'overall_valid':False,'errors':errors,'semantic_grounding':'NOT_ASSESSED'}
        valid=not errors
        result['final_attempts'].append({'answer':answer,'content':message.get('content'),'errors':errors,'valid':valid,'assessment':assessment,'available_research_ids':sorted(available_research),'semantic_grounding':'NOT_ASSESSED'})
        if result['first_valid'] is None: result['first_valid']=valid
        if valid:
            result.update(status='VALID_SIMULATION',final_valid=True,accepted=answer,settings=settings); break
        if arm=='agent' and not revised and turn+1<maxcalls:
            revised=True; result['revision_used']=True
            revision_message={k:copy.deepcopy(message[k]) for k in ['role','content','reasoning_details','reasoning'] if k in message}
            revision_message['role']='assistant';messages.append(revision_message)
            messages.append({'role':'user','content':'One final-only revision is permitted. Correct these validation errors and return the full JSON object. No tools. Removing a citation does not make its associated claim supported. Reassess the associated interpretation, certainty explanation, rationale and limitations; remove unsupported claims or replace them with descriptive case facts. Do not strengthen a claim or invent normal ranges, physiological diagnoses or exclusions. The available research IDs below are the only summaries supplied/retrieved; optional action bases and calculation conventions remain available. A citation-only error does not itself mean the chosen action violates constraints. If an already available applicable source supports the action, correct or remove the incompatible citation without inventing support. You may reconsider the action, but explain any change using case facts; a validation error is not new patient evidence. Available research IDs: '+encode(sorted(available_research))+'. Errors: '+encode(errors)})
        else:
            result['status']='STAFF_REVIEW_REQUIRED'; break
    else:
        result['status']='CALL_LIMIT'
    result['available_research_ids']=sorted(available_research)
    result['semantic_grounding']='NOT_ASSESSED'
    result['elapsed_seconds']=time.monotonic()-started
    result['http_requests']=len(result['attempts'])
    result['providers']=sorted({a['raw']['provider'] for a in result['attempts'] if a['raw'].get('provider')})
    result['usage_records']=[a['raw']['usage'] for a in result['attempts'] if isinstance(a['raw'].get('usage'),dict)]
    costs=[u.get('cost') for u in result['usage_records'] if type(u.get('cost')) in [int,float] and math.isfinite(u['cost']) and u['cost']>=0]
    result['known_cost_usd']=sum(costs)
    result['all_attempt_costs_known']=len(costs)==len(result['attempts']) and bool(costs)
    return result
def execute_batch(client):
    results=[]
    # Validate EVERY existing selected record before sending anything new.
    paths=[(c,arm,repeat,OUT/f"{c['case_id']}_{arm}_r{repeat}.json")
           for repeat in range(1,REPEATS+1) for c in SELECTED for arm in ['baseline','agent']]
    cached={}
    for c,arm,repeat,path in paths:
        if path.exists():
            r=json.loads(path.read_text())
            require(r.get('case_hash')==digest(c) and r.get('fingerprint')==FINGERPRINT,'Cached case/configuration differs; use a new RUN_LABEL.')
            require(r.get('status')!='RUNNING','Interrupted workflow: inspect requests before any new execution.')
            cached[str(path)]=r
    halt=any(r.get('status') in HALT_STATUSES for r in cached.values())
    for c,arm,repeat,path in paths:
        if str(path) in cached:
            results.append(cached[str(path)]);continue
        if halt:continue
        require(client is not None,'Missing client')
        save_json(path,{'status':'RUNNING','case_hash':digest(c),'fingerprint':FINGERPRINT,'started':stamp()})
        print('Running:',c['case_id'],arm,flush=True)
        r=run_workflow(c,arm,client)
        r.update(case_hash=digest(c),repeat=repeat,fingerprint=FINGERPRINT)
        save_json(path,r);results.append(r)
        print(' →',r['status'],'| requests:',r['http_requests'],'| known cost: $',round(r['known_cost_usd'],6),flush=True)
        if r['status'] in HALT_STATUSES:halt=True
    return results


In [ ]:
OFFLINE_CHECKS_PASSED=False
# 11 · Focused offline checks — synthetic fixtures, NOT model results

def fixture_answer(c):
    return {'physiological_interpretation':'Offline fixture; not a model interpretation.',
       'physiology_evidence_ids':[],'certainty':'LOW','certainty_rationale':'Synthetic offline fixture; appropriateness has not been assessed.','activity_assessment':c['context']['reported_activity'],
       'comparisons':comparisons(c),
       'lighting':{'action':'NO_CHANGE','tone':c['room']['tone'],'rationale':'Retain existing permitted lighting.','evidence_ids':[],'policy_ids':['E04']},
       'sound':{'option':'NONE_SILENCE','rationale':'Silence is a permitted prototype choice.','evidence_ids':[],'policy_ids':['E05']},
       'limitations':['Synthetic fixture only.']}
class FakeRouter:
    def __init__(self,messages): self.responses=iter(messages); self.received=[]
    def call(self,messages,use_tools,tag):
        self.received.append((copy.deepcopy(messages),use_tools))
        return {'choice':{'message':next(self.responses),'finish_reason':'stop'},'attempts':[]}

def offline_checks():
    for c in CASES:
        errors,_=validate_answer(fixture_answer(c),c); assert not errors, errors
    c=copy.deepcopy(CASES[0]); c['physiology']['eda_us']['current']=None
    assert comparisons(c)['eda_us']['absolute_change'] is None
    c['physiology']['hr_bpm']['reference']=0
    assert comparisons(c)['hr_bpm']['percent_change'] is None
    bad=fixture_answer(CASES[0]); bad['comparisons']['hr_bpm']['absolute_change']=999
    assert any('COMPARISON_VALUE' in e for e in validate_answer(bad,CASES[0])[0])
    locked=copy.deepcopy(CASES[0]); locked['staff']['lighting_locked']=True
    bad=fixture_answer(locked); bad['lighting']['tone']='WARM'
    assert 'LIGHTING_LOCK' in validate_answer(bad,locked)[0]
    bad=fixture_answer(CASES[0]); bad['sound'].update(option='NATURE_SOUNDS',evidence_ids=['E02'],policy_ids=[])
    restricted=copy.deepcopy(CASES[0]); restricted['staff']['sound_allowed']=False
    assert 'SOUND_RESTRICTION' in validate_answer(bad,restricted)[0]
    c=CASES[0]; signature=[{'type':'reasoning.encrypted','data':'fixture-signature','format':'test','index':0}]
    fake=FakeRouter([{'role':'assistant','content':None,'reasoning_details':signature,
       'tool_calls':[{'id':'t1','type':'function','function':{'name':'calculate_changes','arguments':'{}'}}]},
       {'role':'assistant','content':encode(fixture_answer(c))}])
    result=run_workflow(c,'agent',fake)
    assert result['final_valid'] and len(result['tools'])==1
    assert fake.received[1][0][-2]['reasoning_details']==signature
    bad=fixture_answer(c); bad['activity_assessment']='MOVING'
    fake=FakeRouter([{'content':encode(bad)},{'content':encode(fixture_answer(c))}])
    result=run_workflow(c,'agent',fake)
    assert result['first_valid'] is False and result['final_valid'] and result['revision_used']
    assert fake.received[1][1] is False
    result=run_workflow(c,'baseline',FakeRouter([{'content':encode(bad)}]))
    assert not result['final_valid'] and not result['revision_used']
    result=run_workflow(c,'agent',FakeRouter([{'content':None}]),call_limit=1)
    assert not result['final_valid']
    print('Offline checks passed: case validity, missing/zero values, arithmetic, locks, sound restrictions, native tool continuation, signature preservation, revision and baseline behavior.')
offline_checks()

def evidence_access_checks():
    c=CASES[0]
    a=fixture_answer(c)
    a['lighting'].update(action='DECREASE',evidence_ids=['E01'],policy_ids=[])
    a['sound'].update(option='NATURE_SOUNDS',evidence_ids=['E02'],policy_ids=[])
    content={'role':'assistant','content':encode(a)}
    # Same content is available to the baseline, but not to an agent seeing only an index.
    base=run_workflow(c,'baseline',FakeRouter([content]))
    assert base['first_valid'] and base['final_valid']
    fake=FakeRouter([content,content])
    blocked=run_workflow(c,'agent',fake)
    assert not blocked['first_valid'] and not blocked['final_valid'] and blocked['revision_used']
    assert {'RESEARCH_NOT_AVAILABLE E01','RESEARCH_NOT_AVAILABLE E02'}<=set(blocked['final_attempts'][0]['errors'])
    assert fake.received[1][1] is False  # final-only revision still has no tools
    retrieval={'role':'assistant','content':None,'tool_calls':[{'id':'evidence_1','type':'function',
        'function':{'name':'retrieve_evidence','arguments':encode({'evidence_ids':['E01','E02']})}}]}
    obtained=run_workflow(c,'agent',FakeRouter([retrieval,content]))
    assert obtained['first_valid'] and obtained['available_research_ids']==['E01','E02']
    assert obtained['semantic_grounding']=='NOT_ASSESSED'
    # Research cited only in prose is checked too; seeing E01/E02 does not expose E03.
    inline=copy.deepcopy(a);inline['limitations']=['See E03 for physiological interpretation.']
    blocked=run_workflow(c,'agent',FakeRouter([retrieval,{'content':encode(inline)}]),call_limit=2)
    assert 'RESEARCH_NOT_AVAILABLE E03' in blocked['final_attempts'][0]['errors']
    # A direct answer based entirely on supplied prototype rules remains permitted.
    direct=run_workflow(c,'agent',FakeRouter([{'content':encode(fixture_answer(c))}]))
    assert direct['first_valid'] and not direct['tools']
    # Research/prototype namespaces cannot be interchanged.
    wrong=copy.deepcopy(a);wrong['lighting']['evidence_ids']=['E04']
    assert any(x.startswith('SCHEMA') for x in validate_answer(wrong,c,RESEARCH_IDS)[0])
    wrong=copy.deepcopy(a);wrong['lighting']['policy_ids']=['E04']
    assert 'POLICY_ACTION_MISMATCH lighting/E04' in validate_answer(wrong,c,RESEARCH_IDS)[0]
    # Failed tool execution must not make research available.
    badcall=copy.deepcopy(retrieval);badcall['tool_calls'][0]['function']['arguments']=encode({'evidence_ids':['E99']})
    bad=run_workflow(c,'agent',FakeRouter([badcall]))
    assert bad['status']=='INVALID_TOOL_CALL' and not bad['available_research_ids']
    baseline_input=json.loads(messages_for(c,'baseline')[1]['content'])
    agent_input=json.loads(messages_for(c,'agent')[1]['content'])
    assert all(baseline_input[k]==agent_input[k] for k in ['mandatory_constraints','optional_action_bases','calculation_conventions'])
    assert baseline_input['case']==agent_input['case']
    assert all('finding' not in x for x in agent_input['evidence_index'])
    print('Evidence-access checks passed: unread citations rejected; successful retrieval accepted; inline citations checked; final-only revision cannot bypass access; direct policy-only answers allowed; namespaces and action mappings enforced; failed tools grant no access.')
evidence_access_checks()


def source_role_checks():
    assert len(RESEARCH_IDS)==12 and len(POLICY_IDS)==3
    for entry in EVIDENCE:
        if entry['id'] in RESEARCH_IDS:
            assert all(entry.get(k) for k in ['doi','proposal_key','verified_from','study_design','limitations','ahr_application'])
    c=CASES[0]; a=fixture_answer(c)
    a['lighting']['evidence_ids']=['E11','E12']
    assert not validate_answer(a,c,RESEARCH_IDS)[0]  # relevant context beside policy support
    a['lighting']['policy_ids']=[]
    assert 'NO_APPLICABLE_SUPPORT lighting' in validate_answer(a,c,RESEARCH_IDS)[0]
    a=fixture_answer(c);a['lighting']['evidence_ids']=['E13']
    assert 'EVIDENCE_FIELD_MISMATCH lighting/E13' in validate_answer(a,c,RESEARCH_IDS)[0]
    a=fixture_answer(c);a['physiology_evidence_ids']=['E02']
    assert 'EVIDENCE_FIELD_MISMATCH physiology/E02' in validate_answer(a,c,RESEARCH_IDS)[0]
    c=copy.deepcopy(c);c['preferences']['preferred_sound']='WHITE_NOISE';c['context']['goal']='comfort';c['staff']['allowed_sounds'].append('WHITE_NOISE')
    a=fixture_answer(c);a['sound'].update(option='WHITE_NOISE',evidence_ids=['E09'],policy_ids=[])
    assert 'EVIDENCE_CONTEXT_MISMATCH sound/E09' in validate_answer(a,c,RESEARCH_IDS)[0]
    c['context']['goal']='sleep'
    assert not validate_answer(a,c,RESEARCH_IDS)[0]
    assert not re.search(r'\bE\d+\b',COMMON)  # no named study findings leak into shared instructions
    supplied=json.loads(messages_for(c,'baseline')[1]['content'])['evidence_table']
    retrieved=tool('retrieve_evidence',{'evidence_ids':sorted(RESEARCH_IDS)},c)
    assert supplied==retrieved
    print('Source-role checks passed: metadata coverage, contextual citations, action basis, physiology field, sleep scope, identical summaries and no study-specific prompt leakage.')
source_role_checks()


def policy_dimension_checks():
    c=copy.deepcopy(CASES[0])
    c['staff']['allowed_sounds']=['NATURE_SOUNDS','PREFERRED_CALMING_MUSIC','WHITE_NOISE']  # TEST FIXTURE ONLY
    for sound in ['NATURE_SOUNDS','PREFERRED_CALMING_MUSIC','WHITE_NOISE']:
        c['preferences']['preferred_sound']=sound
        a=fixture_answer(c);a['sound'].update(option=sound,policy_ids=['E05'])
        assert assess_answer(a,c)['overall_valid']
    c['preferences']['preferred_sound']=None
    a=fixture_answer(c);a['sound'].update(option='NATURE_SOUNDS',policy_ids=['E05'])
    assert not assess_answer(a,c)['citation_contract_valid']  # no invented preference support
    a['sound'].update(evidence_ids=['E02'],policy_ids=[])
    assert assess_answer(a,c,RESEARCH_IDS)['overall_valid']  # research route remains available
    c['staff']['sound_allowed']=False
    assert not assess_answer(a,c,RESEARCH_IDS)['action_compliant']
    c=CASES[0];a=fixture_answer(c)
    a['lighting'].update(action='DECREASE',evidence_ids=['E01'],policy_ids=['E04'])
    d=assess_answer(a,c,RESEARCH_IDS)
    assert d['action_compliant'] and d['calculation_valid'] and not d['citation_contract_valid'] and not d['overall_valid']
    a['lighting']['policy_ids']=[]
    assert assess_answer(a,c,RESEARCH_IDS)['overall_valid']
    d=assess_answer(a,c,[])
    assert d['action_compliant'] and not d['citation_contract_valid']
    a['comparisons']['hr_bpm']['absolute_change']=999
    d=assess_answer(a,c,RESEARCH_IDS)
    assert not d['calculation_valid'] and d['action_compliant'] and d['citation_contract_valid']
    d=assess_answer({},c)
    assert d['schema_valid'] is False and d['action_compliant'] is None
    fake=FakeRouter([{'content':encode(fixture_answer(c))}])
    r=run_workflow(c,'agent',fake)
    assert r['final_attempts'][0]['assessment']['overall_valid']
    print('Policy/dimension checks passed: equal preference support, unknown preference, staff prohibition, citation/action/calculation separation and workflow logging.')
policy_dimension_checks()



# Audit regressions: mocked network only, temporary ledgers, no credentials.
from unittest.mock import patch
from tempfile import TemporaryDirectory
import unittest, io
from contextlib import redirect_stdout

class AuditChecks(unittest.TestCase):
    def endpoint(self):
        return {'tag':'fixture-provider','provider_name':PROVIDER,'status':0,'context_length':1048576,
                'max_completion_tokens':65536,'supported_parameters':['tools','tool_choice','reasoning','temperature','max_tokens'],
                'supports_tool_choice':{'auto':True},'pricing':{'prompt':'0.00000010','completion':'0.00000050'}}
    def response(self,body=None,status=200):
        class Response:
            headers={'Retry-After':'0'};text='fixture'
            def json(self):return body
        r=Response();r.status_code=status;return r
    def body(self,cost=.001):
        return {'id':'fixture','model':MODEL,'provider':PROVIDER,'usage':{'cost':cost},
          'choices':[{'finish_reason':'stop','message':{'role':'assistant','content':encode(fixture_answer(CASES[0]))}}]}
    def test_endpoint_filters(self):
        e=self.endpoint();self.assertEqual(len(eligible_endpoints({'id':MODEL,'endpoints':[e]})),1)
        variants=[]
        for key,value in [('supported_parameters',[]),('supports_tool_choice',{'auto':False}),('max_completion_tokens',1024),('context_length',1024),('status',1)]:
            x=copy.deepcopy(e);x[key]=value;variants.append(x)
        for key,value in [('prompt','0.01'),('completion','0.01'),('request','0.1'),('internal_reasoning','0.01')]:
            x=copy.deepcopy(e);x['pricing'][key]=value;variants.append(x)
        for x in variants:
            with self.subTest(x=x),self.assertRaises(ValueError):eligible_endpoints({'id':MODEL,'endpoints':[x]})
    def test_payload_shape(self):
        with TemporaryDirectory() as d:
            r=Router('fixture-key',d,[self.endpoint()]);p=r.payload([],True)
            self.assertEqual(p['tool_choice'],'auto')
            if SEND_TEMPERATURE: self.assertEqual(p['temperature'],1)
            else: self.assertNotIn('temperature',p)
            self.assertEqual(p['provider']['max_price']['completion'],MAX_OUTPUT_PRICE_PER_M)
            self.assertEqual(p['provider']['only'],['fixture-provider'])
            self.assertFalse(p['provider']['allow_fallbacks'])
            self.assertNotIn('tools',r.payload([],False));self.assertNotIn('tool_choice',r.payload([],False))
    def test_no_endpoint_no_call(self):
        with TemporaryDirectory() as d,patch('requests.post') as post:
            with self.assertRaises(ValueError):Router('fixture',d).call([],True,'test')
            post.assert_not_called()
    def test_success_cost_and_secret(self):
        with TemporaryDirectory() as d,patch('requests.post',return_value=self.response(self.body())) as post:
            r=Router('FIXTURE_PRIVATE_KEY',d,[self.endpoint()]);out=r.call([],True,'test')
            self.assertIn('choice',out);self.assertEqual(post.call_count,1);self.assertEqual(r.used(),.001)
            self.assertNotIn('FIXTURE_PRIVATE_KEY',''.join(p.read_text() for p in Path(d).rglob('*.json')))
    def test_embedded_failure_retry(self):
        body={'error':{'code':429,'message':'fixture'}}
        with TemporaryDirectory() as d,patch('requests.post',return_value=self.response(body)) as post,patch('time.sleep'),patch.dict(globals(),RUN_BUDGET_USD=.25):
            r=Router('fixture',d,[self.endpoint()]);out=r.call([],True,'test')
            self.assertEqual(out['error'],'PROVIDER_FAILURE');self.assertEqual(post.call_count,2)
            self.assertTrue(all(not x['cost_known'] for x in r.ledger));self.assertGreater(r.used(),0)
    def test_retry_success(self):
        with TemporaryDirectory() as d,patch('requests.post',side_effect=[self.response({'error':{'code':503}}),self.response(self.body())]) as post,patch('time.sleep'),patch.dict(globals(),RUN_BUDGET_USD=.25):
            r=Router('fixture',d,[self.endpoint()]);out=r.call([],False,'test')
            self.assertIn('choice',out);self.assertEqual(post.call_count,2);self.assertGreater(r.used(),.001)
    def test_timeout_not_retried(self):
        with TemporaryDirectory() as d,patch('requests.post',side_effect=requests.Timeout()) as post:
            r=Router('fixture',d,[self.endpoint()]);out=r.call([],False,'test')
            self.assertEqual(out['error'],'PROVIDER_FAILURE');self.assertEqual(post.call_count,1);self.assertGreater(r.used(),0)
    def test_nontransient_not_retried(self):
        with TemporaryDirectory() as d,patch('requests.post',return_value=self.response({'error':{'code':400}},400)) as post:
            out=Router('fixture',d,[self.endpoint()]).call([],True,'test')
            self.assertEqual(out['error'],'PROVIDER_FAILURE');self.assertEqual(post.call_count,1)
    def test_limits_before_post(self):
        for key,value,status in [('RUN_BUDGET_USD',.000001,'BUDGET_STOP'),('MAX_REQUEST_RESERVATION_USD',.000001,'REQUEST_COST_STOP'),('MAX_HTTP_REQUESTS',0,'REQUEST_LIMIT'),('MAX_ESTIMATED_INPUT_TOKENS',1,'INPUT_SIZE_STOP')]:
            with self.subTest(key=key),TemporaryDirectory() as d,patch.dict(globals(),{key:value}),patch('requests.post') as post:
                out=Router('fixture',d,[self.endpoint()]).call([],False,'test')
                self.assertEqual(out['error'],status);post.assert_not_called()
    def test_ledger_reloaded(self):
        with TemporaryDirectory() as d,patch('requests.post') as post:
            r=Router('fixture',d,[self.endpoint()]);save_json(r.path,[{'accounted_usd':RUN_BUDGET_USD}])
            self.assertEqual(r.call([],False,'test')['error'],'BUDGET_STOP');post.assert_not_called()
    def test_over_reservation_stops(self):
        with TemporaryDirectory() as d,patch('requests.post',return_value=self.response(self.body(MAX_REQUEST_RESERVATION_USD+0.10))):
            out=Router('fixture',d,[self.endpoint()]).call([],False,'test')
            self.assertEqual(out['error'],'REPORTED_COST_EXCEEDED_LOCAL_ESTIMATE')
    def test_invalid_missing_cost_reserved(self):
        for cost in [None,-1,True,'0.1']:
            with self.subTest(cost=cost),TemporaryDirectory() as d,patch('requests.post',return_value=self.response(self.body(cost))):
                r=Router('fixture',d,[self.endpoint()]);r.call([],False,'test')
                self.assertFalse(r.ledger[0]['cost_known']);self.assertGreater(r.used(),0)
    def test_response_shapes_and_identity(self):
        for body,status in [([], 'EMPTY_CHOICES'),({'choices':{}},'EMPTY_CHOICES'),({'choices':[{}]},'MALFORMED_RESPONSE')]:
            with self.subTest(body=body),TemporaryDirectory() as d,patch('requests.post',return_value=self.response(body)):
                self.assertEqual(Router('fixture',d,[self.endpoint()]).call([],False,'test')['error'],status)
        for field,value in [('model','wrong'),('provider','wrong')]:
            body=self.body();body[field]=value
            with TemporaryDirectory() as d,patch('requests.post',return_value=self.response(body)):
                self.assertEqual(Router('fixture',d,[self.endpoint()]).call([],False,'test')['error'],'UPSTREAM_IDENTITY_MISMATCH')
    def test_parse_duplicate_nonfinite(self):
        for text in ['{"a":1,"a":2}','{"a":NaN}','```json\n{}\n```','']:
            with self.subTest(text=text),self.assertRaises(ValueError):parse_final(text)
    def test_bad_native_calls(self):
        for call in ['bad',{'id':'x','type':'function','function':{'name':'json','arguments':'{}'}}]:
            r=run_workflow(CASES[0],'agent',FakeRouter([{'content':None,'tool_calls':[call]}]))
            self.assertEqual(r['status'],'INVALID_TOOL_CALL');self.assertIsNone(r['first_valid'])
    def test_final_revision_signatures_and_limits(self):
        c=CASES[0];a=fixture_answer(c);a['activity_assessment']='MOVING'
        sig=[{'type':'reasoning.encrypted','data':'fixture-signature','format':'test','index':0}]
        fake=FakeRouter([{'content':encode(a),'reasoning_details':sig},{'content':encode(fixture_answer(c))}])
        r=run_workflow(c,'agent',fake)
        self.assertFalse(r['first_valid']);self.assertTrue(r['final_valid']);self.assertFalse(fake.received[1][1])
        self.assertEqual(fake.received[1][0][-2]['reasoning_details'],sig)
        self.assertEqual(len(r['final_attempts']),2)
        self.assertEqual(len(run_workflow(c,'baseline',FakeRouter([{'content':encode(a)}]))['final_attempts']),1)
    def test_tool_and_call_limits(self):
        calls=[{'id':str(i),'type':'function','function':{'name':'calculate_changes','arguments':'{}'}} for i in range(AGENT_MAX_TOOLS+1)]
        r=run_workflow(CASES[0],'agent',FakeRouter([{'tool_calls':calls}]))
        self.assertEqual(r['status'],'TOOL_LIMIT');self.assertFalse(r['tools'])
        messages=[{'tool_calls':[{'id':str(i),'type':'function','function':{'name':'calculate_changes','arguments':'{}'}}]} for i in range(AGENT_MAX_CALLS)]
        fake=FakeRouter(messages);r=run_workflow(CASES[0],'agent',fake)
        self.assertEqual(r['status'],'CALL_LIMIT');self.assertEqual(len(fake.received),AGENT_MAX_CALLS)
    def test_frozen_artifacts_and_lock(self):
        with TemporaryDirectory() as d:
            p=Path(d)/'x.json';write_frozen(p,{'x':1});before=p.read_bytes()
            with self.assertRaises(ValueError):write_frozen(p,{'x':2})
            self.assertEqual(p.read_bytes(),before)
            with generation_lock(d):
                with self.assertRaises(RuntimeError):
                    with generation_lock(d):pass
            self.assertFalse((Path(d)/'generation.lock').exists())
    def test_case_numeric_inputs(self):
        for change in ['nan','extra','silence']:
            c=copy.deepcopy(CASES[0])
            if change=='nan':c['room']['remaining_minutes']=float('nan')
            if change=='extra':c['physiology']['rr_bpm']={}
            if change=='silence':c['room']['volume']=10
            with self.subTest(change=change),self.assertRaises(AssertionError):check_case(c)
    def test_action_boundaries(self):
        c=copy.deepcopy(CASES[0]);c['room']['brightness']=c['staff']['brightness_max']
        a=fixture_answer(c);a['lighting'].update(action='INCREASE',evidence_ids=['E01'],policy_ids=[])
        self.assertIn('NO_INCREASE_POSSIBLE',validate_answer(a,c,RESEARCH_IDS)[0])
        c=copy.deepcopy(CASES[0]);c['staff']['lighting_locked']=True
        a=fixture_answer(c);a['lighting']['tone']='WARM'
        self.assertFalse(assess_answer(a,c)['action_compliant'])
    def test_batch_reuse_and_halt(self):
        with TemporaryDirectory() as d,patch.dict(globals(),{'OUT':Path(d),'SELECTED':[CASES[0]],'REPEATS':1,'FINGERPRINT':'fixture'}):
            good={'content':encode(fixture_answer(CASES[0]))};fake=FakeRouter([good,good])
            first=execute_batch(fake);self.assertEqual(len(first),2);self.assertEqual(len(fake.received),2)
            second=execute_batch(FakeRouter([]));self.assertEqual(first,second)
        with TemporaryDirectory() as d,patch.dict(globals(),{'OUT':Path(d),'SELECTED':[CASES[0]],'REPEATS':1,'FINGERPRINT':'fixture'}):
            path=Path(d)/(CASES[0]['case_id']+'_agent_r1.json')
            save_json(path,{'status':'PROVIDER_FAILURE','case_hash':digest(CASES[0]),'fingerprint':'fixture'})
            fake=FakeRouter([]);self.assertEqual(len(execute_batch(fake)),1);self.assertFalse(fake.received)
    def test_interrupted_record_blocks_all(self):
        with TemporaryDirectory() as d,patch.dict(globals(),{'OUT':Path(d),'SELECTED':[CASES[0]],'REPEATS':1,'FINGERPRINT':'fixture'}):
            save_json(Path(d)/(CASES[0]['case_id']+'_agent_r1.json'),{'status':'RUNNING','case_hash':digest(CASES[0]),'fingerprint':'fixture'})
            fake=FakeRouter([])
            with self.assertRaises(ValueError):execute_batch(fake)
            self.assertFalse(fake.received)

    def test_all_five_cases_both_workflows(self):
        for c in CASES:
            with self.subTest(case=c['case_id']):
                answer=fixture_answer(c)
                # Show a permitted change when sound is allowed, without assuming that it is optimal.
                if not c['context']['procedure']:
                    answer['sound'].update(option='NATURE_SOUNDS',evidence_ids=['E02'],policy_ids=[])
                base=run_workflow(c,'baseline',FakeRouter([{'content':encode(answer)}]))
                self.assertTrue(base['first_valid'])
                calls=[]
                for i,name in enumerate(['calculate_changes','check_data_quality','check_activity_context','retrieve_evidence']):
                    args={'evidence_ids':['E02']} if name=='retrieve_evidence' else {}
                    calls.append({'id':str(i),'type':'function','function':{'name':name,'arguments':encode(args)}})
                agent=run_workflow(c,'agent',FakeRouter([{'content':None,'tool_calls':calls},{'content':encode(answer)}]))
                self.assertTrue(agent['first_valid']);self.assertEqual(len(agent['tools']),4)
                self.assertEqual(base['settings'],agent['settings'])
    def test_citation_repair_can_preserve_actions(self):
        c=CASES[0];a=fixture_answer(c)
        a['lighting'].update(action='DECREASE',tone='WARM',evidence_ids=['E01'],policy_ids=['E04'])
        fixed=copy.deepcopy(a);fixed['lighting']['policy_ids']=[]
        retrieval={'content':None,'tool_calls':[{'id':'x','type':'function','function':{'name':'retrieve_evidence','arguments':encode({'evidence_ids':['E01']})}}]}
        fake=FakeRouter([retrieval,{'content':encode(a)},{'content':encode(fixed)}]);r=run_workflow(c,'agent',fake)
        self.assertFalse(r['first_valid']);self.assertTrue(r['final_valid'])
        self.assertEqual(r['accepted']['lighting']['action'],'DECREASE');self.assertEqual(r['settings']['brightness'],c['room']['brightness']-POLICY['brightness_step'])
        self.assertIn('citation-only error',fake.received[-1][0][-1]['content'])
        self.assertFalse(fake.received[-1][1])
    def test_e04_mapping_unchanged(self):
        e=next(x for x in EVIDENCE if x['id']=='E04');self.assertEqual(e['lighting_actions'],['NO_CHANGE'])
        self.assertFalse(e['mandatory']);self.assertFalse(e['restricts_other_actions'])
        for action in ['INCREASE','DECREASE']:
            a=fixture_answer(CASES[0]);a['lighting']['action']=action
            self.assertIn('POLICY_ACTION_MISMATCH lighting/E04',validate_answer(a,CASES[0])[0])
    def test_certainty_contract_and_access(self):
        a=fixture_answer(CASES[0]);del a['certainty_rationale']
        self.assertFalse(assess_answer(a,CASES[0])['schema_valid'])
        a=fixture_answer(CASES[0]);a['certainty_rationale']='   '
        self.assertFalse(assess_answer(a,CASES[0])['schema_valid'])
        a=fixture_answer(CASES[0]);a['certainty_rationale']='E03 supports my confidence.'
        self.assertIn('RESEARCH_NOT_AVAILABLE E03',validate_answer(a,CASES[0])[0])
        self.assertIn('not a calibrated probability',COMMON)
        # No automatic claim that HIGH is substantively justified.
        a=fixture_answer(CASES[0]);a['certainty']='HIGH'
        self.assertEqual(assess_answer(a,CASES[0])['semantic_grounding'],'NOT_ASSESSED')
    def test_scope_and_missingness(self):
        self.assertEqual(set(CASE_IDS),{'PILOT_201','PILOT_202','PILOT_203','PILOT_204','PILOT_205'})
        self.assertEqual(len(CASE_IDS)*REPEATS*(1+AGENT_MAX_CALLS)*(1+TRANSIENT_RETRIES),MAX_HTTP_REQUESTS)
        c=CASES[2];self.assertEqual(comparisons(c)['eda_us'],{'status':'UNAVAILABLE','absolute_change':None,'percent_change':None})
        for c in CASES:
            for arm in ['baseline','agent']:
                msg=messages_for(c,arm);data=json.loads(msg[1]['content'])
                self.assertEqual(data['case'],c)
                self.assertIn('certainty_rationale',data['OUTPUT_SCHEMA']['required'])
    def test_preferences_never_override_prohibitions(self):
        for sound in ['NATURE_SOUNDS','PREFERRED_CALMING_MUSIC','WHITE_NOISE']:
            for restriction in ['staff','patient','procedure','change_permission']:
                with self.subTest(sound=sound,restriction=restriction):
                    c=copy.deepcopy(CASES[0]);c['preferences']['preferred_sound']=sound
                    if restriction=='staff':c['staff']['sound_allowed']=False
                    if restriction=='patient':c['preferences']['prohibited_sounds']=[sound]
                    if restriction=='procedure':c['context']['procedure']=True;c['staff']['no_sound_during_procedure']=True
                    if restriction=='change_permission':c['preferences']['sound_changes_allowed']=False
                    a=fixture_answer(c);a['sound'].update(option=sound,evidence_ids=[],policy_ids=['E05'])
                    self.assertFalse(assess_answer(a,c)['action_compliant'])

    def test_non_thinking_transport(self):
        with TemporaryDirectory() as d, patch.dict(globals(),REASONING_EFFORT=None):
            e=self.endpoint();e['supported_parameters'].remove('reasoning')
            self.assertEqual(len(eligible_endpoints({'id':MODEL,'endpoints':[e]})),1)
            self.assertNotIn('reasoning',Router('fixture',d,[e]).payload([],True))
        with patch.dict(globals(),REASONING_EFFORT='high'):
            with self.assertRaises(ValueError):eligible_endpoints({'id':MODEL,'endpoints':[e]})
    def test_revision_receives_exact_available_sources(self):
        c=CASES[2];a=fixture_answer(c);a['physiology_evidence_ids']=['E03']
        fake=FakeRouter([{'content':encode(a)},{'content':encode(fixture_answer(c))}])
        r=run_workflow(c,'agent',fake)
        self.assertFalse(r['first_valid']);self.assertTrue(r['final_valid'])
        message=fake.received[1][0][-1]['content']
        self.assertIn('Available research IDs: []',message)
        self.assertIn('Removing a citation does not make its associated claim supported',message)
        self.assertFalse(fake.received[1][1])
    def test_index_is_not_full_evidence(self):
        for c in CASES:
            baseline=json.loads(messages_for(c,'baseline')[1]['content'])
            agent=json.loads(messages_for(c,'agent')[1]['content'])
            self.assertEqual(baseline['case'],agent['case'])
            self.assertEqual(baseline['OUTPUT_SCHEMA'],agent['OUTPUT_SCHEMA'])
            self.assertTrue(all('finding' not in e for e in agent['evidence_index']))
            self.assertTrue(all('finding' in e for e in baseline['evidence_table']))
        self.assertIn('Measurement changes neither confirm nor contradict RESTING',COMMON)
    def test_semantic_review_is_not_silently_scored(self):
        c=CASES[2];a=fixture_answer(c)
        a['physiological_interpretation']='These changes contradict resting and rule out disease.'
        score=assess_answer(a,c,set())
        self.assertTrue(score['overall_valid'])
        self.assertEqual(score['semantic_grounding'],'NOT_ASSESSED')

    def test_optional_support_is_not_a_lighting_lock(self):
        c=CASES[0]
        for action,refs,bases in [('NO_CHANGE',[],['E04']),('DECREASE',['E01'],[])]:
            a=fixture_answer(c);a['lighting'].update(action=action,evidence_ids=refs,policy_ids=bases)
            self.assertTrue(assess_answer(a,c,{'E01'})['overall_valid'])
            locked=copy.deepcopy(c);locked['staff']['lighting_locked']=True
            if action=='DECREASE':self.assertFalse(assess_answer(a,locked,{'E01'})['action_compliant'])
        a=fixture_answer(c);a['lighting'].update(action='DECREASE',evidence_ids=[],policy_ids=['E04'])
        self.assertFalse(assess_answer(a,c,set())['citation_contract_valid'])
    def test_constraint_and_support_channels_are_separate(self):
        for c in CASES:
            for arm in ['baseline','agent']:
                data=json.loads(messages_for(c,arm)[1]['content'])
                self.assertNotIn('prototype_rules',data)
                self.assertEqual(data['mandatory_constraints']['staff'],c['staff'])
                self.assertNotIn('preferred_sound',data['mandatory_constraints']['patient_permissions_and_prohibitions'])
                self.assertEqual({e['id'] for e in data['optional_action_bases']},{'E04','E05'})
                self.assertTrue(all(e['mandatory'] is False and e['restricts_other_actions'] is False for e in data['optional_action_bases']))
                self.assertEqual([e['id'] for e in data['calculation_conventions']],['E06'])
    def test_retrieval_preserves_optional_force(self):
        entries=tool('retrieve_evidence',{'evidence_ids':['E04','E05']},CASES[2])
        self.assertTrue(all(e['role']=='optional_action_basis' and not e['mandatory'] for e in entries))

suite=unittest.defaultTestLoader.loadTestsFromTestCase(AuditChecks)
_test_stream=io.StringIO()
with patch('requests.get',side_effect=AssertionError('Network forbidden in offline checks')),patch('requests.post',side_effect=AssertionError('Unmocked paid request forbidden')):
    with redirect_stdout(io.StringIO()):
        _audit_result=unittest.TextTestRunner(stream=_test_stream,verbosity=1).run(suite)
print(_test_stream.getvalue())
require(_audit_result.wasSuccessful(),'Audit regressions failed; no live calls permitted.')
OFFLINE_CHECKS_PASSED=True
print('Audit checks passed:',_audit_result.testsRun,'named tests (with additional parameterized cases). No network calls.')


In [ ]:
# 11b · Offline first-request and price-cap checks for ALL three model configurations
assert OFFLINE_CHECKS_PASSED and len(CASES)==5 and len(CASE_IDS)*2==10

def activate_model(key, output_folder=None):
    """Set only configuration-specific globals; leave cases, prompts and validator unchanged."""
    global MODEL_CHOICE,P,MODEL,PROVIDER,SEND_TEMPERATURE,REASONING_EFFORT
    global RUN_BUDGET_USD,MAX_REQUEST_RESERVATION_USD,MAX_INPUT_PRICE_PER_M,MAX_OUTPUT_PRICE_PER_M
    global OUT,RUN_LABEL
    assert key in PROFILES and 'deepseek' not in key.lower()
    MODEL_CHOICE=key; P=PROFILES[key]
    MODEL=P['model'];PROVIDER=P['provider_name']; SEND_TEMPERATURE=P['send_temperature']
    REASONING_EFFORT=P['reasoning']
    RUN_BUDGET_USD=P['budget_usd'];MAX_REQUEST_RESERVATION_USD=P['per_request_cap']
    MAX_INPUT_PRICE_PER_M=P['max_input_price_per_m'];MAX_OUTPUT_PRICE_PER_M=P['max_output_price_per_m']
    RUN_LABEL=MASTER_RUN_LABEL+'_'+key
    OUT=Path(output_folder) if output_folder is not None else MASTER_ROOT/key
    OUT.mkdir(parents=True,exist_ok=True)

CHECKED_REQUESTS=[]
for model_key in PROFILES:
    activate_model(model_key,MASTER_ROOT/'shared')
    assert len(CASES)==5
    for c in CASES:
        if c['case_id']=='PILOT_203':
            assert all(x['status']=='UNAVAILABLE' for x in comparisons(c).values())
        if c['case_id']=='PILOT_205':assert c['staff']['lighting_locked'] and not c['context']['procedure']
        if c['case_id']=='PILOT_202':assert c['preferences']['preferred_sound'] is None
        if c['case_id']=='PILOT_204':assert c['context']['reported_activity']=='MOVING'
        for arm in ['baseline','agent']:
            payload=Router('offline-key',MASTER_ROOT/'shared',[{'tag':'offline-dummy','provider_name':PROVIDER}]).payload(messages_for(c,arm),arm=='agent')
            estimate,reserve=Router('offline-key',MASTER_ROOT/'shared',[{'tag':'offline-dummy','provider_name':PROVIDER}]).reservation(payload)
            assert reserve<=MAX_REQUEST_RESERVATION_USD,(model_key,c['case_id'],arm,reserve)
            assert estimate<=MAX_ESTIMATED_INPUT_TOKENS,(model_key,c['case_id'],arm,estimate)
            CHECKED_REQUESTS.append({'model':model_key,'case_id':c['case_id'],'arm':arm,
               'estimated_input_tokens_conservative':estimate,'first_request_reserved_usd':round(reserve,7),
               'provider':PROVIDER,'reasoning':REASONING_EFFORT,'send_temperature':SEND_TEMPERATURE})
activate_model('luna',MASTER_ROOT/'shared')  # shared preflight and offline tests retain their original context
write_frozen(MASTER_ROOT/'shared'/'three_model_first_requests.json',CHECKED_REQUESTS)
assert len(CHECKED_REQUESTS)==30
print('OFFLINE PASS: 5 WESAD source cases x baseline/agent x 3 models = 30 first-call reservation checks.')
display(pd.DataFrame(CHECKED_REQUESTS).groupby('model')['first_request_reserved_usd'].agg(['min','max']).round(6))
print('No paid calls; no API key was accessed by these checks.')


## Execution: three independent model runs, one shared implementation

`RUN_STAGE='offline'` requires **no OpenRouter key**. Run all cells once and inspect the tests. To authorize paid calls, set `RUN_STAGE='smoke'` and `APPROVE_PAID_CALLS=True`, then **Run all**. This tests `PILOT_201` baseline/agent sequentially for **each** enabled model. If any provider is missing or declines required tools/settings, that model stops before or during generation; the others continue.

When you have reviewed smoke results, switch to `RUN_STAGE='full'`, keep `APPROVE_PAID_CALLS=True`, and **Run all again in the same runtime**. Each model must have two saved successful smoke workflows or the master refuses to generate its remaining eight workflows (unless you explicitly change `ALLOW_INVALID_SMOKE_FOR_FULL`). It verifies the frozen model configuration, five source-case SHA-256 hashes, cached workflow hashes and existing ledger before issuing new requests. **No silent repeats of paid runs.**

`USE_GOOGLE_DRIVE=True` is available if enabled from the first offline pass; otherwise Colab files disappear when its runtime resets. You can set `RESTORE_PREVIOUS_ZIP` to the path of the previously exported master ZIP after manually uploading it to the new Colab session. Restore only an archive generated by this notebook, not an arbitrary ZIP. A restore never overwrites existing files, and cached hashes are verified before live execution.

**Provider policy:** one exact endpoint per model, no fallback. The Cerebras–Qwen listing is inconsistent across OpenRouter pages; live endpoint preflight may correctly mark Qwen as unavailable. Provider headquarters is not proof of compute location. Request limit 120 seconds per HTTP call; **no separate overall workflow timeout**. Failures count as failures and are never silently retried beyond the declared one transient retry. No nurse assessments occur here.


In [ ]:
# 12 · Resume-safe multi-model execution. All shared functions above are used unchanged.
# Prior master results, if any, were safely restored in the initialization cell.

# Every model gets a frozen identity and implementation hash. The stage itself is not
# part of the fingerprint so smoke records can be safely reused in the full stage.
def function_contract(fn):
    def constant(x):
        if isinstance(x,types.CodeType): return code(x)
        if isinstance(x,(frozenset,set)):return {'set':sorted([constant(y) for y in x],key=encode)}
        if isinstance(x,tuple):return {'tuple':[constant(y) for y in x]}
        return repr(x)
    def code(c):return {'bytes':c.co_code.hex(),'names':c.co_names,'vars':c.co_varnames,
                         'constants':[constant(x) for x in c.co_consts]}
    return code(fn.__code__)

SOURCE_CASES=[c for c in CASES]
RUN_ERRORS=[]
MODEL_RESULTS={}
FULL_CASES=CASE_IDS[:]

def set_model_contract(model_key):
    global RUN_IDENTITY,CONFIG,FINGERPRINT,SELECTED
    activate_model(model_key)
    SELECTED=[c for c in CASES if c['case_id'] in (CASE_IDS[:1] if RUN_STAGE=='smoke' else CASE_IDS)]
    assert len(SELECTED)==(1 if RUN_STAGE=='smoke' else 5)
    settings={k:globals()[k] for k in ['MODEL','PROVIDER','SEND_TEMPERATURE','CASE_IDS','REPEATS','REASONING_EFFORT',
      'TEMPERATURE','MAX_OUTPUT_TOKENS','AGENT_MAX_CALLS','AGENT_MAX_TOOLS','TRANSIENT_RETRIES','REQUEST_TIMEOUT',
      'PACE_SECONDS','RUN_BUDGET_USD','MAX_REQUEST_RESERVATION_USD','MAX_HTTP_REQUESTS',
      'MAX_ESTIMATED_INPUT_TOKENS','MAX_INPUT_PRICE_PER_M','MAX_OUTPUT_PRICE_PER_M']}
    RUN_IDENTITY={'version':'AHR-WESAD5-master-single-implementation-1.0','profile':model_key,
                  'settings':settings,'global_local_budget_usd':GLOBAL_BUDGET_USD,
                  'external_cases_hash':SOURCE_SHA256_EXPECTED,'source_case_sha256s':SOURCE_CASE_SHA256_EXPECTED}
    write_frozen(OUT/'run_identity.json',RUN_IDENTITY)
    CONFIG={'version':'AHR-WESAD5-master-shared-workflows-1.0','model':MODEL,'provider':PROVIDER,
      'reasoning':REASONING_EFFORT,'temperature':TEMPERATURE if SEND_TEMPERATURE else None,
      'max_output_tokens':MAX_OUTPUT_TOKENS,'agent_max_calls':AGENT_MAX_CALLS,'agent_max_tools':AGENT_MAX_TOOLS,
      'transient_retries':TRANSIENT_RETRIES,'timeout':REQUEST_TIMEOUT,'pace_seconds':PACE_SECONDS,
      'input_price_cap':MAX_INPUT_PRICE_PER_M,'output_price_cap':MAX_OUTPUT_PRICE_PER_M,
      'policy':POLICY,'evidence':EVIDENCE,'prompt':COMMON,'tools':TOOLS,'schema':OUTPUT_SCHEMA,
      'region_basis':P['region_basis'],'run_identity':RUN_IDENTITY}
    CONFIG['implementation']={name:function_contract(fn) for name,fn in
      [('validate',validate_answer),('assess',assess_answer),('case',check_case),('calculate',comparisons),
       ('tools',tool),('prompt',messages_for),('parse',parse_final),('workflow',run_workflow),
       ('endpoint_filter',eligible_endpoints),('transport',Router.call),('payload',Router.payload),
       ('reservation',Router.reservation),('batch',execute_batch)]}
    FINGERPRINT=digest(CONFIG)
    write_frozen(OUT/'experiment_contract.json',{'fingerprint':FINGERPRINT,'configuration':CONFIG,'cases':CASES})
    write_frozen(OUT/'original_source_case_hashes.json',SOURCE_CASE_SHA256_EXPECTED)
    return FINGERPRINT

def check_smoke_gate():
    for arm in ('baseline','agent'):
        path=OUT/f'PILOT_201_{arm}_r1.json'
        require(path.exists(),'Full stage blocked: smoke results are missing for '+MODEL_CHOICE+' / '+arm)
        r=json.loads(path.read_text())
        require(r.get('case_hash')==digest(CASES[0]) and r.get('fingerprint')==FINGERPRINT,
                'Full stage blocked: smoke case/configuration mismatch')
        require(r.get('status')!='RUNNING','Full stage blocked: interrupted smoke; inspect logs')
        require(r.get('final_valid') is True or ALLOW_INVALID_SMOKE_FOR_FULL,
                'Full stage blocked: smoke not fully valid for '+MODEL_CHOICE+' / '+arm)
        require(r.get('status') not in HALT_STATUSES,'Full stage blocked: prior API/budget/provider failure')
    return True

for model_key in ENABLED_MODELS:
    print('\n'+'='*66+'\n',model_key.upper(), 'STAGE:',RUN_STAGE,'\n'+'='*66)
    try:
        set_model_contract(model_key)
        print('Pinned provider:',PROVIDER,'| model:',MODEL,'| per-model budget $',RUN_BUDGET_USD,
              '| global accounted $',round(master_used_usd(),6),'/',GLOBAL_BUDGET_USD)
        require(OFFLINE_CHECKS_PASSED,'Offline checks must complete before any generation')
        if RUN_STAGE=='full':check_smoke_gate()
        if RUN_STAGE=='offline':
            MODEL_RESULTS[model_key]=[json.loads(path.read_text()) for c in CASES for arm in ('baseline','agent')
                         for path in [OUT/f"{c['case_id']}_{arm}_r1.json"] if path.exists()]
            print('OFFLINE. Existing saved results:',len(MODEL_RESULTS[model_key]),'No network.')
        else:
            require(APPROVE_PAID_CALLS,'Explicit paid approval required')
            with generation_lock(OUT):
                # Lightweight no-charge catalogue request; must find the exact approved provider.
                client=prepare_client()
                manifest=OUT/'manifest.json'
                if not manifest.exists():save_json(manifest,{'created':stamp(),'fingerprint':FINGERPRINT,'configuration':CONFIG,'cases':CASES})
                else:require(json.loads(manifest.read_text())['fingerprint']==FINGERPRINT,'Existing manifest mismatch')
                MODEL_RESULTS[model_key]=execute_batch(client)
                print('Finished',model_key,'| saved workflows:',len(MODEL_RESULTS[model_key]))
    except Exception as exc:
        # Print only exception class plus a bounded detail; never include credentials, headers or payloads.
        err={'profile':model_key,'stage':RUN_STAGE,'category':type(exc).__name__,
             'message':str(exc)[:500],'time':stamp()}
        RUN_ERRORS.append(err)
        print('STOPPED / NO SILENT RETRY:',err['category'],err['message'])
        MODEL_RESULTS[model_key]=[json.loads(p.read_text()) for p in (MASTER_ROOT/model_key).glob('PILOT_*_r1.json')
                 if p.is_file() and json.loads(p.read_text()).get('status')!='RUNNING']

if RUN_ERRORS:save_json(MASTER_ROOT/('master_stops_'+RUN_STAGE+'.json'),RUN_ERRORS)
print('\nSaved model results:',{name:len(values) for name,values in MODEL_RESULTS.items()})
print('Local accounted USD across all profiles:',round(master_used_usd(),6),'/',GLOBAL_BUDGET_USD)


In [ ]:
# 13 · Cross-model summary; missing workflows remain NOT_RUN (never convert to failures)
rows=[]; summary=[]; validation_rows=[]
for key in PROFILES:
    model=PROFILES[key]['model']; folder=MASTER_ROOT/key
    ledger_path=folder/'cost_ledger.json'
    ledger=json.loads(ledger_path.read_text()) if ledger_path.exists() else []
    cost_confirmed=sum(v['accounted_usd'] for v in ledger if v.get('cost_known') is True)
    cost_accounted=sum(v['accounted_usd'] for v in ledger)
    statuses=[]; valid_count=0; first_count=0; attempted=0; provider_failure=0
    prior_stops=[]
    for stop_path in MASTER_ROOT.glob('master_stops_*.json'):
        prior_stops += [e for e in json.loads(stop_path.read_text()) if e.get('profile')==key]
    for c in CASES:
        for arm in ('baseline','agent'):
            path=folder/f"{c['case_id']}_{arm}_r1.json"
            r=json.loads(path.read_text()) if path.exists() else {}
            status=r.get('status','NOT_RUN')
            if status!='NOT_RUN':
                attempted+=1
                if r.get('final_valid') is True:valid_count+=1
                if r.get('first_valid') is True:first_count+=1
                if status in HALT_STATUSES:provider_failure+=1
            a=r.get('accepted') or {}
            rows.append({'model_key':key,'model':model,'fixed_provider':PROFILES[key]['provider_name'],
               'case_id':c['case_id'],'workflow':arm,'status':status,
               'first_valid':r.get('first_valid'),'final_valid':r.get('final_valid'),
               'revision_used':r.get('revision_used'),'http_requests':r.get('http_requests'),
               'elapsed_seconds':r.get('elapsed_seconds'),'known_workflow_cost_usd':r.get('known_cost_usd'),
               'lighting_action':a.get('lighting',{}).get('action'),'sound_option':a.get('sound',{}).get('option'),
               'reported_providers':';'.join(r.get('providers') or []),
               'semantic_grounding':'NOT_ASSESSED'})
            for i,answer_attempt in enumerate(r.get('final_attempts',[])):
                a=answer_attempt.get('assessment') or {}
                validation_rows.append({'model_key':key,'case_id':c['case_id'],'workflow':arm,'stage':'FIRST' if i==0 else 'REVISION',
                        **{k:a.get(k) for k in ['schema_valid','calculation_valid','action_compliant','citation_contract_valid','overall_valid']},
                        'validation_errors':';'.join(answer_attempt.get('errors',[]))})
    summary.append({'model_key':key,'model':model,'fixed_provider':PROFILES[key]['provider_name'],
       'cases_planned':5,'workflows_planned':10,'workflows_attempted':attempted,
       'valid_first_answer':first_count,'valid_final':valid_count,
       'nonvalid_attempted':attempted-valid_count,'provider_or_budget_stops':provider_failure,
       'not_run':10-attempted,'http_attempts':len(ledger),
       'known_cost_usd':round(cost_confirmed,8),'accounted_usd_with_reservations':round(cost_accounted,8),
       'assessment':'TECHNICAL_ONLY',
       'recorded_run_stops':'; '.join((e['stage']+': '+e['category']+': '+e['message'][:100]) for e in prior_stops)})
RESULT_TABLE=pd.DataFrame(rows)
MODEL_COMPARISON=pd.DataFrame(summary)
VALIDATION_TABLE=pd.DataFrame(validation_rows)
display(MODEL_COMPARISON)
display(RESULT_TABLE)
RESULT_TABLE.to_csv(MASTER_ROOT/'workflow_results.csv',index=False)
MODEL_COMPARISON.to_csv(MASTER_ROOT/'model_comparison.csv',index=False)
if validation_rows:VALIDATION_TABLE.to_csv(MASTER_ROOT/'validation_dimensions.csv',index=False)
print('Count comparison: denominators distinguish attempted, valid, provider-failed and NOT_RUN.')
print('Known charges omit unknown-cost attempts; accounted amount includes conservative reservations.')


## Important interpretation

**This is exclusively a DEVELOPMENT reliability screen.** The comparative table records how often each model completes the same agentic and non-agentic workflows, along with structural, calculation, tool, evidence-reference and constraint compliance, cost, latency and upstream failures. A technically valid JSON recommendation is **not** necessarily clinically appropriate. Neither nurse assessments nor the planned 30 held-out cases are included here.

The existing Qwen3-235B comparison achieved 9/10 valid original-WESAD workflows with its controlled four-provider routing. That result remains a separate historical record, not an additional row generated by the current model calls.


In [ ]:
# 14 · Researcher audit instructions only (no nurse form/scoring in this development screen)
REVIEW_NOTES={
    'research_stage':'DEVELOPMENT / MODEL SELECTION',
    'objective':'Reliably complete the baseline and agentic workflows using identical original WESAD-derived cases.',
    'measure':'first-response validity; final validity; schema/calculation/constraint/evidence-reference compliance; provider failures; cost and latency',
    'not_measured':'Independent nurse alignment, recommendation appropriateness, clinical efficacy or patient outcomes',
    'fixed_cases':CASE_IDS,
    'models':{key:{'model':p['model'],'provider':p['provider_name'],'reasoning':p['reasoning'],
                     'send_temperature':p['send_temperature']} for key,p in PROFILES.items()},
    'changes_policy':'Do not repair or rescore failed provider/model outcomes. Any prompt/provider/model parameter change starts a new version and separate run identity.',
    'privacy':'WESAD is research physiology; patient preferences, room settings and context are synthetic overlays.'
}
write_frozen(MASTER_ROOT/'researcher_audit_scope.json',REVIEW_NOTES)
print('Technical reliability outputs produced. No nurse scoring performed.')


In [ ]:
# 15 · Save complete master export; never export credentials, model API key or Authorization headers
archive=BASE/(MASTER_RUN_LABEL+'_RESULTS.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in MASTER_ROOT.rglob('*'):
        if path.is_file() and path.name!='generation.lock':
            z.write(path,path.relative_to(MASTER_ROOT).as_posix())
assert zipfile.ZipFile(archive).testzip() is None,'Results ZIP checksum verification failed.'
print('Results archive:',archive,'| bytes:',archive.stat().st_size)
print('This ZIP contains model responses, logs, original case sources, summaries and evidence, never the OpenRouter key.')
if DOWNLOAD_ZIP and RUN_LIVE:
    try:
        from google.colab import files
        files.download(str(archive))
    except ImportError:
        print('Use your notebook environment to download:',archive)


## After the run

Download **`AHR_WESAD5_Three_Model_Master_v1_RESULTS.zip`** and upload it back to the research chat. It includes:

- `model_comparison.csv` — final-valid versus first-valid, attempted versus unattempted, API failures and per-model costs;
- `workflow_results.csv` — all 30 intended case × model × workflow combinations, including `NOT_RUN` rows;
- `validation_dimensions.csv` — first-answer versus revision validation checks;
- `luna/`, `gemini/`, `qwen/` — raw model outputs, requests, responses, endpoint-preflight metadata and individual cost ledgers;
- `shared/` — original WESAD inputs, manifest, normalization audit, evidence and deterministic offline checks.

**No offline fixture can be counted as a live LLM completion.** A Qwen/Cerebras `NO COMPATIBLE ENDPOINT` result is a legitimate screening observation and must not be silently replaced by a different provider. Review current OpenRouter pricing, service quotas and provider geography before interpreting costs and regional restrictions. The $7.25 combined local ceiling is **not** a hard OpenRouter account-level budget.
